# VoltRelay — Battery-Swap Network Diagnostic
### Data Analytics Hackathon submission

**Scope:** 3.88M swap attempts, 1.49M station-hours of telemetry, 20k riders, 6.5k battery packs,
44k support tickets, 152 stations, 6 cities, Jan 2024 – Jun 2025.

**Notebook contents**
1. Setup & configuration
2. Data understanding (schema, integrity, quality traps)
3. Cleaning with an auditable ledger
4. Q1 — How is the network actually performing?
5. Q2 — What breaks a swap? (failure taxonomy + driver model)
6. Q3 — What drives rider churn?
7. Q4 — What erodes margin?
8. Q5 — Anomalies worth flagging separately
9. Deep dives — pricing pilot (DiD), competitor entry (DiD), battery-bank solvency
10. Q6 — Prioritised recommendations, impact-sized

Every number quoted in the accompanying report is regenerated here and written to
`reports/key_findings.json`.

## 0. Colab bootstrap — mount Drive and locate the dataset

**Run this cell first.** It is safe to run anywhere.

* **In Google Colab:** mounts your Drive, then finds the folder that holds the eight CSVs
  and points `VOLTRELAY_DATA` at it. Nothing is hard-coded to a local machine.
* **Locally:** it detects that it is not in Colab and does nothing.

If your dataset is not in `MyDrive`, either move it there under the name below, or paste a
shareable folder link into `COLAB_SHARED_FOLDER_URL` and it will be downloaded with gdown.

In [1]:
COLAB_DATASET_FOLDER = "Hackathon Data Set | Gradient"   # folder name inside MyDrive
COLAB_SHARED_FOLDER_URL = ""                             # optional: https://drive.google.com/drive/folders/...

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

if IN_COLAB:
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception as exc:                      # pragma: no cover - notebook UX
        print("Drive mount skipped:", exc)

    if not os.environ.get("VOLTRELAY_DATA"):
        candidates = [
            Path("/content/drive/MyDrive") / COLAB_DATASET_FOLDER,
            Path("/content/drive/MyDrive/datasets") / COLAB_DATASET_FOLDER,
            Path("/content/drive/MyDrive/Colab Notebooks") / COLAB_DATASET_FOLDER,
            Path("/content") / COLAB_DATASET_FOLDER,
            Path("/content/data"),
        ]
        found = next((c for c in candidates if (c / "swap_events.csv").exists()), None)
        if found is not None:
            os.environ["VOLTRELAY_DATA"] = str(found)
            print("dataset found:", found)
        elif COLAB_SHARED_FOLDER_URL:
            import gdown
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)
            import gdown
            gdown.download_folder(COLAB_SHARED_FOLDER_URL, output="/content/dataset",
                                  quiet=False, use_cookies=False)
            os.environ["VOLTRELAY_DATA"] = "/content/dataset"
            print("dataset downloaded to /content/dataset")
        else:
            print("Dataset not found in Drive.\n"
                  f"  Option A: put the eight CSVs in MyDrive/{COLAB_DATASET_FOLDER}/\n"
                  "  Option B: paste a shareable Drive folder link into COLAB_SHARED_FOLDER_URL above.")

print("running in Colab:", IN_COLAB)
print("VOLTRELAY_DATA   :", os.environ.get("VOLTRELAY_DATA", "(not set - will search locally)"))

Mounted at /content/drive
Dataset not found in Drive.
  Option A: put the eight CSVs in MyDrive/Hackathon Data Set | Gradient/
  Option B: paste a shareable Drive folder link into COLAB_SHARED_FOLDER_URL above.
running in Colab: True
VOLTRELAY_DATA   : (not set - will search locally)


## 1. Setup & configuration

The cell below finds the dataset whether you are in Colab (mounted Drive), in a local
checkout, or pointing at a custom folder with `VOLTRELAY_DATA`.

In [3]:
import json
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

%matplotlib inline

# ---- locate the data -------------------------------------------------------------
def find_data_dir():
    cand = []
    env = os.environ.get("VOLTRELAY_DATA")
    if env:
        cand.append(Path(env))
    cand.append(Path.cwd() / "data")
    cand.append(Path.cwd())
    cand += [Path("/content"), Path("/content/drive/MyDrive"), Path("/content/drive/MyDrive/datasets")]
    tried = []
    for c in cand:
        if not c.exists():
            continue
        tried.append(str(c))
        if (c / "swap_events.csv").exists() or (c / "swap_events.csv.gz").exists():
            return c
        # one level down, e.g. data/Hackathon Data Set | Gradient/
        for sub in sorted(p for p in c.iterdir() if p.is_dir()):
            if (sub / "swap_events.csv").exists() or (sub / "swap_events.csv.gz").exists():
                return sub
    raise FileNotFoundError(
        "Could not find swap_events.csv. Looked in:\n  " + "\n  ".join(tried) +
        "\nSet os.environ['VOLTRELAY_DATA'] to the folder that holds the eight CSVs."
    )

DATA = find_data_dir()
ROOT = Path.cwd()
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
(ROOT / "reports").mkdir(exist_ok=True)
print("data dir :", DATA)

# ---- house plotting style --------------------------------------------------------
INK = "#12212e"
MUTED = "#697a89"
GRID = "#dfe6ec"
C_MAIN = "#1f6feb"
C_BAD = "#e0553b"
C_GOOD = "#1a9e7a"
C_WARN = "#e2a53a"
C_ALT = "#7d5ba6"

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 160, "savefig.bbox": "tight",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRID, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 10.5,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

# Explicit figure registry: numbers are pinned here so inserting a new chart later cannot
# renumber the ones already cited by the report, README and dashboard.
FIGNUM = {
    "q1_kpi_trend": 1, "q1_failure_mix": 2, "q2_stock_ladder": 3, "q2_thermal": 4,
    "q2_battery_supplier": 5, "q2_odds_ratios": 6, "q2_tickets_vs_failure": 7,
    "q3_retention": 8, "q3_simpson": 9, "q3_panel_churn": 10,
    "q4_station_margin": 11, "q4_discount": 12,
    "q9_pricing_did": 13, "q9_competitor_did": 14, "q9_battery_solvency": 15,
    "q11_cooling_targets": 16, "q10_priorities": 17,
}
FIGN = {"n": 0}


def savefig(fig, name, note=None):
    """Save a figure once and return its path. Numbering comes from FIGNUM, not call order."""
    FIGN["n"] += 1
    num = FIGNUM.get(name, 80 + FIGN["n"])
    if note:
        fig.text(0.005, -0.02, note, fontsize=7.5, color=MUTED, ha="left", va="top")
    path = FIG / f"{num:02d}_{name}.png"
    fig.savefig(path)
    return path


def inr(v, unit="auto"):
    """Indian-format money so headline numbers read naturally."""
    a = abs(v)
    if unit == "auto":
        if a >= 1e7:
            return f"₹{v/1e7:,.2f} Cr"
        if a >= 1e5:
            return f"₹{v/1e5:,.1f} L"
        if a >= 1e3:
            return f"₹{v/1e3:,.1f} K"
        return f"₹{v:,.0f}"
    if unit == "cr":
        return f"₹{v/1e7:,.2f} Cr"
    if unit == "l":
        return f"₹{v/1e5:,.1f} L"
    return f"₹{v:,.0f}"


def lakh_fmt(x, _=None):
    return f"{x/1e5:,.0f}L" if abs(x) >= 1e5 else f"{x:,.0f}"


FINDINGS = {}


def record(key, value):
    FINDINGS[key] = value
    return value


def table(df, title=None, note=None):
    """Print a table with a caption instead of a bare repr."""
    if title:
        print(f"\n■ {title}")
    if note:
        print(f"  ({note})")
    print(df.to_string())
    print()

FileNotFoundError: Could not find swap_events.csv. Looked in:
  /content
  /content
  /content/drive/MyDrive
Set os.environ['VOLTRELAY_DATA'] to the folder that holds the eight CSVs.

## 2. Data understanding

Eight tables, one grain each. Before any analysis: shapes, key uniqueness, referential
integrity, date coverage, and the null structure of every column.

In [ ]:
FILES = {
    "swaps": "swap_events.csv",
    "hourly": "station_hourly_status.csv",
    "riders": "riders.csv",
    "batteries": "batteries.csv",
    "tickets": "support_tickets.csv",
    "stations": "stations.csv",
    "context": "city_daily_context.csv",
    "partners": "fleet_partners.csv",
}


def read(name, **kw):
    f = DATA / FILES[name]
    if not f.exists() and (DATA / (FILES[name] + ".gz")).exists():
        f = DATA / (FILES[name] + ".gz")
    return pd.read_csv(f, **kw)


SCHEMA_ROWS = []
for name, fn in FILES.items():
    head = read(name, nrows=20000)
    SCHEMA_ROWS.append({"table": name, "file": fn, "cols": head.shape[1],
                        "sample_rows": head.shape[0]})
table(pd.DataFrame(SCHEMA_ROWS), "Table inventory")

### 2.1 Column-level null profile
The two fact tables are large, so we profile a random 400k-row sample of swaps and every
row of everything else. Nulls here are *meaningful*, not random — we carry that forward.

In [ ]:
swaps_head = read("swaps", nrows=400_000)
prof = pd.DataFrame({
    "dtype": swaps_head.dtypes.astype(str),
    "null_%": (100 * swaps_head.isna().mean()).round(2),
    "n_unique_sample": swaps_head.nunique(),
})
table(prof, "swap_events column profile (400k-row sample)")

### 2.2 Referential integrity and key uniqueness
Every foreign key resolves. That is unusual for real data and it tells us the failures we
see are *operational*, not data-entry garbage.

In [ ]:
stations = read("stations", parse_dates=["commissioned_date", "decommissioned_date",
                                        "firmware_updated_date", "competitor_within_1_5km_since"])
riders = read("riders", parse_dates=["signup_date"])
batteries = read("batteries", parse_dates=["manufacture_date", "commission_date", "retired_date"])
partners = read("partners", parse_dates=["contract_start_date", "amendment_date"])
context = read("context", parse_dates=["date"])
tickets = read("tickets", parse_dates=["created_ts"])
hourly = read("hourly", parse_dates=["hour_start"])

INTEGRITY = []


def check(label, child, parent, table_name):
    have = child.dropna()
    orphan = int((~have.isin(set(parent.dropna()))).sum())
    INTEGRITY.append({"relationship": label, "non_null": len(have), "orphans": orphan,
                      "orphan_%": round(100 * orphan / max(len(have), 1), 4)})


se_ids = read("swaps", usecols=["rider_id", "station_id", "battery_in_id", "battery_out_id"])
for lbl, col, par, nm in [
    ("swaps.rider_id → riders", se_ids.rider_id, riders.rider_id, "riders"),
    ("swaps.station_id → stations", se_ids.station_id, stations.station_id, "stations"),
    ("swaps.battery_in_id → batteries", se_ids.battery_in_id, batteries.battery_id, "batteries"),
    ("swaps.battery_out_id → batteries", se_ids.battery_out_id, batteries.battery_id, "batteries"),
    ("hourly.station_id → stations", hourly.station_id, stations.station_id, "stations"),
    ("tickets.rider_id → riders", tickets.rider_id, riders.rider_id, "riders"),
    ("tickets.station_id → stations", tickets.station_id, stations.station_id, "stations"),
    ("tickets.battery_id → batteries", tickets.battery_id, batteries.battery_id, "batteries"),
    ("riders.partner_id → partners", riders.partner_id, partners.partner_id, "partners"),
]:
    check(lbl, col, par, nm)
table(pd.DataFrame(INTEGRITY), "Foreign-key integrity")

del se_ids
swaps_ids = read("swaps", usecols=["event_id"])
print(f"swap event_id unique: {swaps_ids.event_id.is_unique}  ({len(swaps_ids):,} rows)")
print(f"station_hourly composite key unique: "
      f"{not hourly.duplicated(['station_id', 'hour_start']).any()}")
print(f"riders.rider_id unique: {riders.rider_id.is_unique}   "
      f"batteries.battery_id unique: {batteries.battery_id.is_unique}")
del swaps_ids

### 2.3 Date coverage
Note the honesty check on `support_tickets`: it starts in March 2023, ten months before the
swap window, and runs one day past it. Tickets outside the swap window must be dropped before
any rate calculation or the ticket rate is inflated.

In [ ]:
COVER = pd.DataFrame([
    {"table": "swap_events", "min": read("swaps", usecols=["event_ts"]).event_ts.min(),
     "max": read("swaps", usecols=["event_ts"]).event_ts.max()},
    {"table": "station_hourly_status", "min": hourly.hour_start.min(), "max": hourly.hour_start.max()},
    {"table": "riders.signup_date", "min": riders.signup_date.min(), "max": riders.signup_date.max()},
    {"table": "batteries.manufacture_date", "min": batteries.manufacture_date.min(),
     "max": batteries.manufacture_date.max()},
    {"table": "support_tickets", "min": tickets.created_ts.min(), "max": tickets.created_ts.max()},
    {"table": "city_daily_context", "min": context.date.min(), "max": context.date.max()},
])
table(COVER, "Date coverage per table")

## 3. Cleaning

Six issues matter, and each is recorded with its row impact so the analysis is auditable.
Three are **traps** that would silently corrupt conclusions if left alone.

In [ ]:
LEDGER = []


def ledger(step, before, after, why):
    LEDGER.append({"step": step, "rows_before": before, "rows_after": after,
                   "rows_affected": before - after, "rationale": why})

# ---- 3.1 typed load of the fact table -------------------------------------------
SWAP_DTYPES = {
    "event_id": "str", "rider_id": "category", "station_id": "category",
    "event_type": "category", "queue_wait_sec": "int32",
    "battery_in_id": "category", "battery_out_id": "category",
    "soc_in_pct": "float32", "soh_in_pct": "float32",
    "soc_out_pct": "float32", "soh_out_pct": "float32",
    "km_since_last_swap": "float32", "tariff_code": "category",
    "list_price_inr": "float32", "discount_inr": "float32",
    "amount_charged_inr": "float32", "energy_to_recharge_kwh": "float32",
    "station_firmware": "category", "sync_mode": "category",
}
n_raw = sum(1 for _ in open(DATA / FILES["swaps"] if (DATA / FILES["swaps"]).exists()
                            else DATA / (FILES["swaps"] + ".gz"))) - 1
USE = sorted(set(list(SWAP_DTYPES) + ["event_ts"]))
sw = read("swaps", usecols=USE, dtype=SWAP_DTYPES, parse_dates=["event_ts"])
ledger("typed load", n_raw, len(sw), "explicit dtypes keep peak memory ~1.6 GB")

sw["date"] = sw.event_ts.dt.normalize()
sw["month"] = sw.event_ts.dt.to_period("M")
sw["hour"] = sw.event_ts.dt.hour.astype("int8")
sw["dow"] = sw.event_ts.dt.dayofweek.astype("int8")
sw["is_weekend"] = (sw.dow >= 5).astype("int8")
sw["completed"] = (sw.event_type == "swap_completed").astype("int8")
sw["failed"] = 1 - sw.completed
sw["net_price"] = sw.amount_charged_inr
sw["discount_pct"] = 100 * sw.discount_inr / sw.list_price_inr.replace(0, np.nan)

# ---- 3.2 exact duplicate events ---------------------------------------------------
before = len(sw)
sw = sw.drop_duplicates(subset=["event_id"]).copy()
ledger("drop duplicate event_id", before, len(sw), "event_id is the declared primary key")

# ---- 3.3 impossible battery telemetry on completed swaps -------------------------
before = len(sw)
ok = sw.completed == 1
sw.loc[ok & (sw.battery_out_id.isna()), "battery_out_id"] = pd.NA  # keep, flag instead
soc_bad = ok & ((sw.soc_out_pct > 100) | (sw.soh_out_pct > 100))
sw.loc[soc_bad, ["soc_out_pct", "soh_out_pct"]] = np.nan
ledger("null SoC/SoH > 100% on completed swaps", before, len(sw),
       f"{int(soc_bad.sum()):,} sensor readings exceed physical max — clipped to NaN, not deleted")

# ---- 3.4 negative distance since last swap --------------------------------------
neg_km = (sw.km_since_last_swap < 0)
sw.loc[neg_km, "km_since_last_swap"] = np.nan
ledger("negative km_since_last_swap → NaN", len(sw), len(sw),
       f"{int(neg_km.sum()):,} rows (odometer reset / clock skew), preserved as rows")

# ---- 3.5 dead columns -------------------------------------------------------------
dead = [c for c in ["attempt_seq", "payment_mode"] if c in sw.columns]
print("constant columns removed from analysis:", dead)
print("  attempt_seq is identically 1 for all 3.88M rows → no retry chain can be reconstructed")
print("  payment_mode is 'partner_invoice' for every row, including independent pay-as-you-go riders")
print("  → billing attribution cannot be trusted; we price from tariff_code + amounts instead")

# ---- 3.6 rider city normalisation -------------------------------------------------
CITY_MAP = {
    "BLR": "Bengaluru", "Bangalore": "Bengaluru", "bengaluru": "Bengaluru",
    "Bombay": "Mumbai", "MUM": "Mumbai",
    "Delhi": "Delhi NCR", "New Delhi": "Delhi NCR", "Gurgaon": "Delhi NCR",
    "HYD": "Hyderabad", "Hyd": "Hyderabad", "hyderabad": "Hyderabad",
    "JAI": "Jaipur", "jaipur": "Jaipur",
    "PUN": "Pune", "pune": "Pune",
}
riders["home_city_raw"] = riders.home_city
riders["home_city"] = riders.home_city.replace(CITY_MAP)
n_norm = int((riders.home_city != riders.home_city_raw).sum())
ledger("normalise rider city spellings", len(riders), len(riders),
       f"{n_norm} riders re-mapped from {riders.home_city_raw.nunique()} raw spellings to 6 cities")

# ---- 3.7 tickets outside the swap window -----------------------------------------
T0, T1 = sw.event_ts.min(), sw.event_ts.max()
before = len(tickets)
tickets = tickets[(tickets.created_ts >= T0) & (tickets.created_ts <= T1)].copy()
ledger("tickets inside swap window", before, len(tickets),
       f"{before - len(tickets)} tickets fall outside {T0.date()}–{T1.date()}")

table(pd.DataFrame(LEDGER), "Cleaning ledger")

### 3.1 The three analytic traps

**Trap 1 — the "healthy battery" illusion.** Failure rows have no `battery_out_id`, because no
pack was dispensed. Any "success rate by out-pack SOH" therefore returns 100% for every SOH
band. That is survivorship, not safety. We use the *in-pack* (returned) SOH for battery effects.

In [ ]:
out_soh = (sw.assign(band=pd.cut(sw.soh_out_pct, [0, 65, 75, 85, 95, 101]))
             .groupby("band", observed=True)
             .agg(attempts=("completed", "size"), success_pct=("completed", lambda s: 100 * s.mean())))
table(out_soh, "TRAP: success rate by out-pack SOH — every band is 100%",
      "because only completed swaps have a battery_out_id")

**Trap 2 — templated ticket comments.** 44,000 tickets contain only 89 distinct comment strings,
and each string maps to exactly one category. Text mining would rediscover the category column.
We report this and analyse categories instead — no fake NLP.

In [ ]:
print(f"distinct rider_comment: {tickets.rider_comment.nunique()} for {len(tickets):,} tickets")
print(f"comments mapping to >1 category: {(tickets.groupby('rider_comment').category.nunique() > 1).sum()}")

**Trap 3 — CSAT is missing not-at-random.** CSAT exists only when a ticket was resolved;
duplicates and unresolved tickets have none. Mean CSAT therefore measures the resolved
population only.

In [ ]:
table(pd.crosstab(tickets.resolution_status, tickets.csat_score.notna(),
                  rownames=["resolution"], colnames=["csat present"]),
      "TRAP: CSAT missingness follows resolution status")

## 4. Q1 — How is the network actually performing?

Headline KPIs, then the two questions leadership actually asked: is growth costing us
service quality, and is the network getting better or worse over time?

In [ ]:
kpi_month = sw.groupby("month", observed=True).agg(
    attempts=("completed", "size"),
    swaps=("completed", "sum"),
    riders=("rider_id", "nunique"),
    stations=("station_id", "nunique"),
    revenue=("net_price", "sum"),
    list_value=("list_price_inr", "sum"),
    discount=("discount_inr", "sum"),
    energy_kwh=("energy_to_recharge_kwh", "sum"),
).reset_index()
kpi_month["success_pct"] = 100 * kpi_month.swaps / kpi_month.attempts
kpi_month["swaps_per_rider"] = kpi_month.swaps / kpi_month.riders
kpi_month["net_price"] = kpi_month.revenue / kpi_month.swaps
kpi_month["discount_pct"] = 100 * kpi_month.discount / kpi_month.list_value
kpi_month["month"] = kpi_month.month.astype(str)
table(kpi_month.round(2), "Monthly network KPIs")

HEAD = {
    "window": f"{T0.date()} → {T1.date()}",
    "attempts": int(len(sw)),
    "swaps": int(sw.completed.sum()),
    "success_pct": round(100 * sw.completed.mean(), 2),
    "revenue": float(sw.net_price.sum()),
    "riders_total": int(riders.rider_id.nunique()),
    "riders_active": int(sw.rider_id.nunique()),
    "stations": int(stations.station_id.nunique()),
    "cities": int(stations.city.nunique()),
    "swaps_per_rider_mean": round(float(sw.groupby("rider_id", observed=True).size().mean()), 1),
    "median_swap_monthly_riders": int(kpi_month.riders.median()),
}
record("headline", HEAD)
print(json.dumps(HEAD, indent=2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13.2, 7.4))
x = np.arange(len(kpi_month))
lbl = kpi_month.month.str[2:]

ax = axes[0, 0]
ax.bar(x, kpi_month.swaps / 1000, color=C_MAIN, width=0.72, label="completed swaps")
ax.plot(x, kpi_month.attempts / 1000, color=C_BAD, lw=1.8, marker="o", ms=3, label="attempts")
ax.set_title("Volume keeps compounding (+191% swaps Jan-24 → Jun-25)")
ax.set_ylabel("thousand")
ax.set_xticks(x[::3]); ax.set_xticklabels(lbl[::3], rotation=0)
ax.legend(loc="upper left")

ax = axes[0, 1]
ax.plot(x, kpi_month.success_pct, color=C_MAIN, lw=2.2, marker="o", ms=3.5)
ax.axhline(kpi_month.success_pct.median(), color=MUTED, ls=":", lw=1)
ax.fill_between(x, 86, kpi_month.success_pct, color=C_MAIN, alpha=0.08)
for i in (4, 5, 16, 17):
    ax.scatter([i], [kpi_month.success_pct.iloc[i]], color=C_BAD, zorder=5, s=46)
ax.annotate("summer 2024\n87.7%", (4.1, 88.6), fontsize=8.5, color=C_BAD)
ax.annotate("summer 2025\n90.1%", (16.0, 90.4), fontsize=8.5, color=C_BAD)
ax.set_title("…while success rate is flat at ~95.6% except in summer")
ax.set_ylabel("success rate (%)")
ax.set_xticks(x[::3]); ax.set_xticklabels(lbl[::3])
ax.set_ylim(86, 97)

ax = axes[1, 0]
ax.plot(x, kpi_month.revenue / 1e7, color=C_GOOD, lw=2.2, marker="o", ms=3.5)
ax.set_title("Revenue ₹6.3 Cr → ₹20.0 Cr per month")
ax.set_ylabel("₹ crore / month")
ax.set_xticks(x[::3]); ax.set_xticklabels(lbl[::3])

ax = axes[1, 1]
ax.plot(x, kpi_month.riders / 1000, color=C_ALT, lw=2.2, marker="o", ms=3.5)
ax.set_title("Active riders 4.2k → 12.0k, still accelerating")
ax.set_ylabel("thousand active riders")
ax.set_xticks(x[::3]); ax.set_xticklabels(lbl[::3])

fig.suptitle("Q1 · Network performance, Jan 2024 – Jun 2025", x=0.02, ha="left",
             fontsize=15, fontweight="bold", color=INK)
fig.tight_layout(rect=[0, 0, 1, 0.96])
savefig(fig, "q1_kpi_trend", "Source: swap_events, 3,877,013 attempts. Success rate = completed / attempts.")
plt.show()

### 4.1 Growth is not the problem — summer capacity is

Success rate sits at 95.6% ± 0.1pp for **eleven of eighteen months**. It collapses only in
Apr–Jun of both years. That pattern is seasonal and systemic, not a slow decay from growth.
The next cell tests whether the growth in volume explains it.

In [ ]:
sw["summer"] = sw.month.dt.month.isin([4, 5, 6])
seas = sw.groupby(["summer", sw.month.dt.year], observed=True).agg(
    attempts=("completed", "size"), success_pct=("completed", lambda s: 100 * s.mean()),
    no_batt=("event_type", lambda s: 100 * (s == "failed_no_charged_battery").mean()),
    abandon=("event_type", lambda s: 100 * (s == "abandoned_queue").mean()),
)
seas.index.names = ["summer", "year"]
table(seas.round(2), "Success rate by season and year")

non_summer = sw[~sw.summer]
print(f"non-summer success rate: {100*non_summer.completed.mean():.2f}% over {len(non_summer):,} attempts")
print(f"non-summer monthly volatility of success rate: "
      f"{non_summer.groupby('month', observed=True).completed.mean().std()*100:.2f} pp")
record("seasonality", {
    "nonsummer_success_pct": round(100 * non_summer.completed.mean(), 2),
    "summer24_success_pct": round(100 * sw[(sw.summer) & (sw.event_ts.dt.year == 2024)].completed.mean(), 2),
    "summer25_success_pct": round(100 * sw[(sw.summer) & (sw.event_ts.dt.year == 2025)].completed.mean(), 2),
})

### 4.2 The failure taxonomy of the whole network
Four ways to fail, and they are not interchangeable: two are supply-side (no charged pack,
queue abandonment), two are demand/technical (rider cancels, system error).

In [ ]:
tax = sw.event_type.value_counts().rename("events").to_frame()
tax["share_of_failures_%"] = np.where(tax.index == "swap_completed", np.nan,
                                      100 * tax.events / tax.events.drop("swap_completed").sum())
table(tax.round(2), "Swap attempt outcomes across the full window")

fail_mix = (sw[sw.completed == 0]
            .groupby(["month", "event_type"], observed=True).size().unstack().fillna(0))
fail_mix.index = fail_mix.index.astype(str)
table((100 * fail_mix.div(fail_mix.sum(axis=1), axis=0)).round(2),
      "Share of failures by type, % per month")

fig, ax = plt.subplots(figsize=(10.4, 4.3))
order = ["failed_no_charged_battery", "abandoned_queue", "cancelled_by_rider", "failed_system_error"]
colors = [C_BAD, C_WARN, MUTED, C_ALT]
bottom = np.zeros(len(fail_mix))
for t, c in zip(order, colors):
    ax.bar(np.arange(len(fail_mix)), fail_mix[t], bottom=bottom, label=t.replace("_", " "),
           color=c, width=0.78)
    bottom += fail_mix[t].values
ax.set_xticks(np.arange(len(fail_mix)))
ax.set_xticklabels(fail_mix.index.str[2:], rotation=45, ha="right")
ax.set_ylabel("failed attempts")
ax.set_title("Failures are supply-side in summer and technical the rest of the year")
ax.legend(ncol=4, loc="upper left", fontsize=8.5)
fig.tight_layout()
savefig(fig, "q1_failure_mix", "Failed attempts per month by event_type.")
plt.show()

**Read this:** system errors are 0.29–0.31% of attempts *every single month* — a flat technical
floor. Everything that moves is `no_charged_battery` and `abandoned_queue`, and they move
together and seasonally. That is a charging-capacity story, not a software story.

In [ ]:
rec = fail_mix.sum()
record("failure_taxonomy", {k: int(v) for k, v in tax.events.items()})
record("failure_shares_of_failures", {k: round(100 * v / rec.sum(), 2) for k, v in rec.items()})
_se_share = 100 * fail_mix.failed_system_error / kpi_month.attempts.values
record("system_error_share_range", [round(float(_se_share.min()), 2),
                                    round(float(_se_share.max()), 2)])
print("system_error rows per month, min/max:",
      int(fail_mix.failed_system_error.min()), int(fail_mix.failed_system_error.max()))
print(f"system errors as a share of attempts that month: {_se_share.min():.2f}% to {_se_share.max():.2f}%")

## 5. Q2 — What breaks a swap?

We work from mechanism outward: a swap fails when the cabinet cannot hand over a charged pack
fast enough. Three candidate mechanisms — **stock**, **heat/thermal derating**, **hardware
generation** — plus battery condition. Then a model that holds the others constant, so we can
say which is primary.

In [ ]:
# attach the hourly operating state to each attempt
hourly["swap_hour"] = hourly.hour_start.dt.floor("h")
H = hourly[["station_id", "swap_hour", "charged_2w_min", "charged_2w_avg", "packs_charging",
            "chargers_online", "cabinet_temp_c", "ambient_temp_c", "outage_minutes",
            "telemetry_status", "avg_charge_minutes", "grid_kwh"]].copy()

sw["swap_hour"] = sw.event_ts.dt.floor("h")
swh = sw.merge(H, on=["station_id", "swap_hour"], how="left", suffixes=("", "_h"))
swh = swh.merge(stations[["station_id", "city", "zone", "location_type", "host_type",
                          "charger_generation", "connectivity_tier", "expansion_wave",
                          "slots_2w", "slots_3w", "inventory_target_2w", "grid_tariff_inr_kwh",
                          "monthly_rent_inr", "monthly_maintenance_inr", "firmware_version"]],
                on="station_id", how="left")
swh = swh.merge(context, left_on=["city", "date"], right_on=["city", "date"], how="left")
print(f"hourly join coverage: {100*swh.charged_2w_min.notna().mean():.2f}% of attempts")
print(f"context join coverage: {100*swh.heat_alert.notna().mean():.2f}% of attempts")
print(f"merged frame: {swh.shape[0]:,} rows x {swh.shape[1]} cols")
del H

### 5.1 Mechanism 1 — charged stock is the tightest constraint
`charged_2w_min` is the lowest count of charged 2W packs present in the station during that hour.
It is the operational definition of "can I actually get a battery right now".

In [ ]:
swh["stock_band"] = pd.cut(swh.charged_2w_min, [-0.1, 0, 1, 3, 6, 10, 100],
                           labels=["0 (empty)", "1", "2–3", "4–6", "7–10", "11+"])
stock_tab = swh.groupby("stock_band", observed=True).agg(
    attempts=("completed", "size"),
    success_pct=("completed", lambda s: 100 * s.mean()),
    no_batt_pct=("event_type", lambda s: 100 * (s == "failed_no_charged_battery").mean()),
    abandon_pct=("event_type", lambda s: 100 * (s == "abandoned_queue").mean()),
    median_queue_sec=("queue_wait_sec", "median"),
).reset_index()
table(stock_tab.round(2), "Failure rate by charged-pack availability in the swap hour")

fig, ax = plt.subplots(figsize=(9.6, 4.3))
ax2 = ax.twinx()
ax.bar(np.arange(len(stock_tab)), stock_tab.no_batt_pct, color=C_BAD, width=0.62,
       label="no charged battery (%)")
ax2.plot(np.arange(len(stock_tab)), stock_tab.success_pct, color=INK, lw=2, marker="o", ms=5,
         label="overall success rate (%)")
ax.set_xticks(np.arange(len(stock_tab)))
ax.set_xticklabels(stock_tab.stock_band)
ax.set_xlabel("lowest charged-2W stock in the hour")
ax.set_ylabel("failures: no charged battery (%)", color=C_BAD)
ax2.set_ylabel("success rate (%)", color=INK)
ax2.grid(False)
ax.set_title("A station with zero charged packs fails 3.5× more often")
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.72))
ax2.legend(loc="lower right")
fig.tight_layout()
savefig(fig, "q2_stock_ladder", "stock_band from station_hourly_status.charged_2w_min, 99.3% of attempts matched.")
plt.show()

record("stock_ladder", stock_tab.round(3).to_dict(orient="records"))

### 5.2 Mechanism 2 — heat degrades the cabinet, and the cabinet is where the bottleneck is
On heat-alert days success falls to 85.1%. Cabinet temperature is the mediating variable: above
45 °C the chargers cannot keep packs cool enough to fast-charge, so recharge takes longer and
stock runs out.

In [ ]:
heat_tab = swh.groupby("heat_alert", observed=True).agg(
    attempts=("completed", "size"), success_pct=("completed", lambda s: 100 * s.mean()),
    avg_charge_min=("avg_charge_minutes", "mean"),
    pct_stock_zero=("charged_2w_min", lambda s: 100 * (s == 0).mean()),
    cabinet_temp=("cabinet_temp_c", "mean"),
)
table(heat_tab.round(2), "Heat-alert days vs normal days")

swh["temp_band"] = pd.cut(swh.cabinet_temp_c, [0, 25, 30, 35, 40, 45, 50, 70],
                          labels=["<25", "25–30", "30–35", "35–40", "40–45", "45–50", "50+"])
temp_tab = swh.groupby("temp_band", observed=True).agg(
    attempts=("completed", "size"),
    success_pct=("completed", lambda s: 100 * s.mean()),
    charge_min=("avg_charge_minutes", "mean"),
    pct_stock_zero=("charged_2w_min", lambda s: 100 * (s == 0).mean()),
).reset_index()
table(temp_tab.round(2), "Success by cabinet temperature")

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.2))
ax = axes[0]
ax.bar(np.arange(len(temp_tab)), temp_tab.success_pct, color=[C_GOOD] * 3 + [C_WARN] * 2 + [C_BAD] * 2,
       width=0.64)
for i, v in enumerate(temp_tab.success_pct):
    ax.text(i, v + 0.15, f"{v:.1f}", ha="center", fontsize=8.5, color=INK)
ax.set_xticks(np.arange(len(temp_tab))); ax.set_xticklabels(temp_tab.temp_band)
ax.set_ylim(84, 97)
ax.set_xlabel("cabinet temperature (°C)"); ax.set_ylabel("success rate (%)")
ax.set_title("Success falls 8.6 pp from coolest to hottest cabinet")

ax = axes[1]
ax.plot(np.arange(len(temp_tab)), temp_tab.charge_min, color=C_BAD, lw=2.2, marker="o", ms=5,
        label="mean recharge time (min)")
ax.set_xticks(np.arange(len(temp_tab))); ax.set_xticklabels(temp_tab.temp_band)
ax.set_xlabel("cabinet temperature (°C)"); ax.set_ylabel("minutes")
ax2 = ax.twinx(); ax2.grid(False)
ax2.bar(np.arange(len(temp_tab)), temp_tab.pct_stock_zero, color=C_ALT, alpha=0.35, width=0.5,
        label="% of attempts in a zero-stock hour")
ax2.set_ylabel("% attempts with zero stock", color=C_ALT)
ax.set_title("Hot cabinets recharge slower → more empty-stock hours")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8.5)
fig.tight_layout()
savefig(fig, "q2_thermal", "Cabinet temp from station_hourly_status; success from swap_events.")
plt.show()

record("heat", {
    "heat_days_success_pct": round(float(heat_tab.loc[True, "success_pct"]), 2),
    "normal_days_success_pct": round(float(heat_tab.loc[False, "success_pct"]), 2),
    "heat_attempts": int(heat_tab.loc[True, "attempts"]),
    "coolest_success_pct": round(float(temp_tab.success_pct.iloc[0]), 2),
    "hottest_success_pct": round(float(temp_tab.success_pct.iloc[-1]), 2),
    "charge_min_cool": round(float(temp_tab.charge_min.iloc[0]), 1),
    "charge_min_hot": round(float(temp_tab.charge_min.iloc[-1]), 1),
})

### 5.3 Mechanism 3 — charger generation and station design
Gen1 cabinets fail ~2.9 pp more often than Gen2. But Gen1 stations are concentrated in the older
launch-wave cities (Jaipur, Delhi), so the raw gap conflates hardware with location. Section 5.5
separates them.

In [ ]:
gen_tab = swh.groupby("charger_generation", observed=True).agg(
    attempts=("completed", "size"),
    success_pct=("completed", lambda s: 100 * s.mean()),
    charge_min=("avg_charge_minutes", "mean"),
    pct_stock_zero=("charged_2w_min", lambda s: 100 * (s == 0).mean()),
    stations=("station_id", "nunique"),
)
table(gen_tab.round(2), "By charger generation")

conn_tab = swh.groupby("connectivity_tier", observed=True).agg(
    attempts=("completed", "size"), success_pct=("completed", lambda s: 100 * s.mean()),
    syserr_pct=("event_type", lambda s: 100 * (s == "failed_system_error").mean()),
    offline_batch_pct=("sync_mode", lambda s: 100 * (s == "offline_batch").mean()),
)
table(conn_tab.round(2), "By connectivity tier")

stat_tab = (swh.groupby("station_id", observed=True)
            .agg(attempts=("completed", "size"),
                 success_pct=("completed", lambda s: 100 * s.mean()),
                 no_batt_pct=("event_type", lambda s: 100 * (s == "failed_no_charged_battery").mean()),
                 abandon_pct=("event_type", lambda s: 100 * (s == "abandoned_queue").mean()),
                 syserr_pct=("event_type", lambda s: 100 * (s == "failed_system_error").mean()),
                 median_queue=("queue_wait_sec", "median"),
                 revenue=("net_price", "sum"))
            .merge(stations, on="station_id", how="left"))
print(f"station success rate: median {stat_tab.success_pct.median():.2f}%, "
      f"IQR {stat_tab.success_pct.quantile(0.25):.2f}–{stat_tab.success_pct.quantile(0.75):.2f}%, "
      f"range {stat_tab.success_pct.min():.2f}–{stat_tab.success_pct.max():.2f}%")

city_tab = stat_tab.groupby("city").agg(
    stations=("station_id", "size"), attempts=("attempts", "sum"),
    success_pct=("success_pct", "median"), no_batt_pct=("no_batt_pct", "median"),
    revenue=("revenue", "sum"))
table(city_tab.round(2).sort_values("success_pct"), "By city (median across stations)")

worst = (stat_tab[stat_tab.attempts >= 5000]
         .nsmallest(10, "success_pct")[["station_id", "city", "zone", "attempts", "success_pct",
                                        "no_batt_pct", "abandon_pct", "median_queue",
                                        "charger_generation", "connectivity_tier", "slots_2w"]])
table(worst.round(2).reset_index(drop=True), "Ten worst stations (≥5,000 attempts)")

record("station_spread", {
    "median_success_pct": round(float(stat_tab.success_pct.median()), 2),
    "best": round(float(stat_tab.success_pct.max()), 2),
    "worst": round(float(stat_tab.success_pct.min()), 2),
    "gen1_success": round(float(gen_tab.loc["Gen1", "success_pct"]), 2),
    "gen2_success": round(float(gen_tab.loc["Gen2", "success_pct"]), 2),
})

### 5.4 Mechanism 4 — battery condition
With the survivorship trap handled, the informative variable is the SOH of the pack the rider
*returns*. Packs below 70% SOH are associated with ~5 pp lower success — degraded packs are
likelier to be rejected, quarantined, or to fault mid-swap.

In [ ]:
swh["in_soh_band"] = pd.cut(swh.soh_in_pct, [0, 65, 70, 75, 80, 85, 90, 101],
                            labels=["<65", "65–70", "70–75", "75–80", "80–85", "85–90", "90+"])
soh_tab = swh.groupby("in_soh_band", observed=True).agg(
    attempts=("completed", "size"),
    success_pct=("completed", lambda s: 100 * s.mean()),
    syserr_pct=("event_type", lambda s: 100 * (s == "failed_system_error").mean()),
)
table(soh_tab.round(3), "Success by returned-pack SOH")

bsup = batteries.set_index("battery_id")
bat_panel = (swh[["battery_in_id", "completed"]].dropna()
             .assign(battery_in_id=lambda d: d.battery_in_id.astype(str))
             .merge(bsup[["supplier", "pack_type", "manufacturing_lot", "bms_firmware"]],
                    left_on="battery_in_id", right_index=True, how="left"))
sup_tab = bat_panel.groupby("supplier").agg(
    attempts=("completed", "size"), success_pct=("completed", lambda s: 100 * s.mean()))
sup_tab["packs"] = batteries.groupby("supplier").size()
sup_tab["median_soh"] = batteries.groupby("supplier").current_soh_pct.median()
sup_tab["retired"] = batteries[batteries.retired_date.notna()].groupby("supplier").size()
table(sup_tab.fillna(0).round(2), "By battery supplier")

_deliv = (sw[sw.completed == 1]
          .assign(bid=lambda d: d.battery_out_id.astype(str))
          .groupby("bid").size())
age_deg = (bsup.assign(age_days=(pd.Timestamp("2025-06-30") - bsup.commission_date).dt.days)
           .assign(deliv=lambda d: d.index.map(_deliv).fillna(0))
           .assign(drop=lambda d: d.initial_soh_pct - d.current_soh_pct))
deg_tab = age_deg.groupby("supplier").agg(
    packs=("drop", "size"), median_deliveries=("deliv", "median"),
    median_soh_drop=("drop", "median"), median_current_soh=("current_soh_pct", "median"))
deg_tab["soh_drop_per_100_deliveries"] = (100 * deg_tab.median_soh_drop /
                                          deg_tab.median_deliveries).round(2)
table(deg_tab.round(2), "Degradation by supplier — Kyron packs are the outlier")
record("battery_supplier", deg_tab.round(3).reset_index().to_dict(orient="records"))

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.2))
ax = axes[0]
sup_order = deg_tab.sort_values("median_current_soh").index
data = [age_deg.loc[age_deg.supplier == s, "current_soh_pct"].dropna() for s in sup_order]
try:  # matplotlib >= 3.9 renamed labels -> tick_labels
    bp = ax.boxplot(data, tick_labels=sup_order, widths=0.55, patch_artist=True, showfliers=False)
except TypeError:
    bp = ax.boxplot(data, labels=sup_order, widths=0.55, patch_artist=True, showfliers=False)
for patch, c in zip(bp["boxes"], [C_BAD, C_WARN, C_GOOD]):
    patch.set_facecolor(c); patch.set_alpha(0.55); patch.set_edgecolor(INK)
for med in bp["medians"]:
    med.set_color(INK); med.set_linewidth(1.6)
ax.axhline(70, color=C_BAD, ls="--", lw=1.2)
ax.text(0.52, 70.8, "70% SOH ≈ end-of-useful-life", fontsize=8, color=C_BAD)
ax.set_ylabel("current SOH (%)"); ax.set_title("Kyron packs sit far below the fleet")
ax.grid(axis="x", visible=False)

ax = axes[1]
for s, c in zip(sup_order, [C_BAD, C_WARN, C_GOOD][::-1] if len(sup_order) == 3 else [C_MAIN] * len(sup_order)):
    d = age_deg[age_deg.supplier == s]
    ax.scatter(d.deliv, d.current_soh_pct, s=7, alpha=0.35, color=c, label=s, linewidths=0)
ax.set_xlabel("completed deliveries per pack"); ax.set_ylabel("current SOH (%)")
ax.set_title("Degradation per delivery, not per calendar day")
ax.legend(markerscale=3)
fig.tight_layout()
savefig(fig, "q2_battery_supplier", "batteries.csv joined to deliveries from swap_events (completed only).")
plt.show()

### 5.5 Which driver is primary? A cell-aggregated logistic model

Marginal rates confound each other: hot days are also busy days, Gen1 stations are also in the
oldest cities. So we model failure probability with all factors at once. We aggregate to
predictor cells and fit a weighted binomial GLM (IRLS) — exact, and it avoids loading a
3.9M-row design matrix.

Reported as **odds ratios with 95% CIs**. Adding stock availability (M3) is the mediation test:
if heat's effect collapses once stock is controlled, heat acts *through* the charging bottleneck.

In [ ]:
def fit_logit(X, y, w, iters=40, ridge=1e-7):
    """Weighted binomial IRLS. X has an intercept column. Returns beta, se."""
    beta = np.zeros(X.shape[1])
    for _ in range(iters):
        eta = X @ beta
        mu = 1.0 / (1.0 + np.exp(-eta))
        W = w * mu * (1 - mu)
        z = eta + (y - mu) / np.maximum(mu * (1 - mu), 1e-9)
        XtW = X.T * W
        A = XtW @ X + ridge * np.eye(X.shape[1])
        beta_new = np.linalg.solve(A, XtW @ z)
        if np.max(np.abs(beta_new - beta)) < 1e-9:
            beta = beta_new
            break
        beta = beta_new
    eta = X @ beta
    mu = 1.0 / (1.0 + np.exp(-eta))
    W = w * mu * (1 - mu)
    cov = np.linalg.inv(X.T * W @ X + ridge * np.eye(X.shape[1]))
    return beta, np.sqrt(np.diag(cov))


def or_table(beta, se, names, ref_flags):
    out = pd.DataFrame({"term": names, "odds_ratio": np.exp(beta),
                        "lo": np.exp(beta - 1.96 * se), "hi": np.exp(beta + 1.96 * se)})
    out.loc[ref_flags, ["odds_ratio", "lo", "hi"]] = 1.0
    return out


swh["hour_block"] = pd.cut(swh.hour, [-1, 5, 10, 15, 19, 23],
                           labels=["00–05", "06–10", "11–15", "16–19", "20–23"])
swh["month_idx"] = ((swh.event_ts.dt.year - 2024) * 12 + swh.event_ts.dt.month).astype("int16")
swh["heat_f"] = swh.heat_alert.fillna(False).astype(int)
swh["tband"] = swh.temp_band.astype(str)
swh["sband"] = swh.stock_band.astype(str)
swh["soh_band_s"] = swh.in_soh_band.astype(str)
swh["ok"] = swh.completed.astype(int)
swh["bad"] = 1 - swh.ok

CELL_KEYS = ["heat_f", "tband", "sband", "soh_band_s", "charger_generation",
             "connectivity_tier", "hour_block", "city", "month_idx"]
cells = (swh.groupby(CELL_KEYS, observed=True)
         .agg(attempts=("ok", "size"), failures=("bad", "sum")).reset_index())
cells = cells[cells.attempts > 0]
print(f"model cells: {len(cells):,} covering {cells.attempts.sum():,} attempts")


def design(df, terms):
    parts = [np.ones((len(df), 1))]
    names = ["intercept"]
    for col, ref in terms:
        lv = sorted(df[col].astype(str).unique())
        for v in lv:
            if v == ref:
                continue
            parts.append((df[col].astype(str) == v).astype(float).values.reshape(-1, 1))
            names.append(f"{col}={v}")
    return np.hstack(parts), names, [n == "intercept" for n in names]


BASE = [("heat_f", "0"), ("hour_block", "00–05"), ("city", "Bengaluru")]
M1 = BASE
M2 = M1 + [("tband", "<25"), ("charger_generation", "Gen2"), ("connectivity_tier", "good"),
           ("soh_band_s", "90+")]
M3 = M2 + [("sband", "11+")]

fits = {}
for tag, terms in [("M1 · time+city only", M1), ("M2 · +hardware, heat, battery", M2),
                   ("M3 · +charged stock (mediation)", M3)]:
    X, names, refs = design(cells, terms)
    b, s = fit_logit(X, (cells.failures / cells.attempts).values, cells.attempts.values)
    fits[tag] = or_table(b, s, names, refs)
    print(f"fitted {tag}: {X.shape[1]} parameters")

keys = ["heat_f=1", "tband=40–45", "tband=45–50", "tband=50+", "sband=0 (empty)", "sband=1",
        "sband=2–3", "charger_generation=Gen1", "charger_generation=Gen3",
        "connectivity_tier=poor", "soh_band_s=<65", "soh_band_s=65–70"]
rows = []
for k in keys:
    r = {"term": k}
    for tag, t in fits.items():
        row = t[t.term == k]
        if len(row):
            r[tag] = f"{row.odds_ratio.iloc[0]:.2f} [{row.lo.iloc[0]:.2f}–{row.hi.iloc[0]:.2f}]"
        else:
            r[tag] = "—"
    rows.append(r)
or_show = pd.DataFrame(rows)
table(or_show, "Odds ratios for failure (95% CI), the same terms across three models",
      "OR > 1 = higher failure odds than the reference level")

heat_or = {}
for tag, t in fits.items():
    heat_or[tag] = round(float(t.loc[t.term == "heat_f=1", "odds_ratio"].iloc[0]), 3)
record("heat_odds_ratio_by_model", heat_or)
print("\nMediation read: heat-day odds ratio is", heat_or)

fig, ax = plt.subplots(figsize=(9.4, 5.2))
show = ["heat_f=1", "tband=45–50", "tband=50+", "sband=0 (empty)", "sband=2–3",
        "charger_generation=Gen1", "connectivity_tier=poor", "soh_band_s=<65"]
t3 = fits["M3 · +charged stock (mediation)"].set_index("term")
t1 = fits["M1 · time+city only"].set_index("term")
yy = np.arange(len(show))[::-1]
for y, k in zip(yy, show):
    if k in t1.index and not (t1.loc[k, "lo"] == 1 and t1.loc[k, "hi"] == 1):
        ax.plot([t1.loc[k, "lo"], t1.loc[k, "hi"]], [y + 0.16] * 2, color=MUTED, lw=2, alpha=0.8)
        ax.scatter(t1.loc[k, "odds_ratio"], y + 0.16, color=MUTED, s=34, zorder=3,
                   label="time + city only" if y == yy[0] else None)
    if k in t3.index:
        ax.plot([t3.loc[k, "lo"], t3.loc[k, "hi"]], [y - 0.16] * 2, color=C_BAD, lw=2)
        ax.scatter(t3.loc[k, "odds_ratio"], y - 0.16, color=C_BAD, s=34, zorder=3,
                   label="full model" if y == yy[0] else None)
ax.axvline(1, color=INK, lw=1)
ax.set_yticks(yy); ax.set_yticklabels([s.replace("=", " = ").replace("_", " ") for s in show])
ax.set_xscale("log")
ax.set_xlabel("odds ratio for a failed swap (log scale)")
ax.set_title("After controlling for everything measurable, stock and heat dominate")
ax.legend(loc="lower right")
fig.tight_layout()
savefig(fig, "q2_odds_ratios", "Cell-aggregated weighted binomial GLM, 3.88M attempts.")
plt.show()

### 5.6 Ticket evidence confirms the mechanism
Support tickets are an independent channel: if the failure rate is real, riders complain about
the same thing. `no_battery_available` is the single largest category, and the station-level
correlation between ticket rate and success rate is −0.63.

In [ ]:
tk = tickets.copy()
tk["month"] = tk.created_ts.dt.to_period("M")
cat_month = tk.pivot_table(index="month", columns="category", values="ticket_id",
                           aggfunc="size").fillna(0)
cat_month.index = cat_month.index.astype(str)
table(cat_month.astype(int), "Ticket volume by category and month")

tstat = swh.groupby("station_id", observed=True).agg(
    attempts=("completed", "size"), success_pct=("completed", lambda s: 100 * s.mean()))
tstat = tstat.join(tk.dropna(subset=["station_id"]).groupby("station_id").size().rename("tickets"))
tstat["tickets"] = tstat.tickets.fillna(0)
tstat["tickets_per_1k"] = 1000 * tstat.tickets / tstat.attempts
r_tk = tstat.success_pct.corr(tstat.tickets_per_1k)
print(f"corr(station success rate, tickets per 1k attempts) = {r_tk:.3f}")

fig, ax = plt.subplots(figsize=(8.4, 4.4))
ax.scatter(tstat.success_pct, tstat.tickets_per_1k, s=np.sqrt(tstat.attempts) / 3.2,
           alpha=0.55, color=C_MAIN, edgecolor="white", linewidth=0.5)
z = np.polyfit(tstat.success_pct, tstat.tickets_per_1k, 1)
xs = np.linspace(tstat.success_pct.min(), tstat.success_pct.max(), 20)
ax.plot(xs, np.polyval(z, xs), color=C_BAD, lw=1.8)
ax.set_xlabel("station success rate (%)"); ax.set_ylabel("tickets per 1,000 attempts")
ax.set_title(f"Riders complain where swaps fail (r = {r_tk:.2f}); bubble = station volume")
fig.tight_layout()
savefig(fig, "q2_tickets_vs_failure", f"Pearson r = {r_tk:.3f} across 152 stations.")
plt.show()

record("tickets", {
    "top_category": tk.category.value_counts().index[0],
    "top_category_n": int(tk.category.value_counts().iloc[0]),
    "no_batt_tickets": int((tk.category == "no_battery_available").sum()),
    "corr_station_success_vs_tickets": round(float(r_tk), 3),
    "median_resolution_hours": round(float(tk.resolution_hours.median()), 1),
    "resolved_pct": round(100 * (tk.resolution_status == "resolved").mean(), 1),
})

---
### Working-set management
The Q2 merge is the widest frame in this notebook (~3.9M attempts × 60 columns). Everything later
sections need from it is cached here as small aggregates, then the merge is released, so the whole
notebook also finishes inside a 3 GB runtime.

In [ ]:
import gc

HEAT_ATTEMPTS = int((swh.heat_alert == True).sum())
HEAT_SUCC = float(swh.loc[swh.heat_alert == True, "completed"].mean() * 100)
NOHEAT_SUCC = float(swh.loc[swh.heat_alert == False, "completed"].mean() * 100)
OFFLINE_ATTEMPTS = int((swh.sync_mode == "offline_batch").sum())
OFFLINE_SHARE = float((swh.sync_mode == "offline_batch").mean())

CTX_TABLE = []
for flag, label in [("heat_alert", "heat alert"), ("flood_disruption", "flood disruption"),
                    ("is_public_holiday", "public holiday"),
                    ("competitor_promo_active", "competitor promo"),
                    ("ecommerce_sale_event", "e-commerce sale")]:
    t = swh.loc[swh[flag] == True, "completed"]
    c = swh.loc[swh[flag] == False, "completed"]
    CTX_TABLE.append({"condition": label, "attempts": int(len(t)),
                      "success_true_pct": round(100 * float(t.mean()), 2),
                      "success_false_pct": round(100 * float(c.mean()), 2),
                      "delta_pp": round(100 * (float(t.mean()) - float(c.mean())), 2)})
_hi = swh.loc[swh.rainfall_mm > 25, "completed"]
_lo = swh.loc[swh.rainfall_mm <= 25, "completed"]
CTX_TABLE.append({"condition": "heavy rain (>25 mm)", "attempts": int(len(_hi)),
                  "success_true_pct": round(100 * float(_hi.mean()), 2),
                  "success_false_pct": round(100 * float(_lo.mean()), 2),
                  "delta_pp": round(100 * (float(_hi.mean()) - float(_lo.mean())), 2)})
CTX_TABLE = pd.DataFrame(CTX_TABLE)

OFF_TABLE = (swh.groupby("sync_mode", observed=True)
             .agg(attempts=("completed", "size"),
                  success_pct=("completed", lambda s: 100 * s.mean()),
                  syserr_pct=("event_type", lambda s: 100 * (s == "failed_system_error").mean())))
CONN_OFFLINE = (swh.groupby("connectivity_tier", observed=True)
                .agg(offline_share=("sync_mode", lambda s: 100 * (s == "offline_batch").mean())))
FW_TABLE = (swh.groupby(["firmware_version", "charger_generation"], observed=True)
            .agg(attempts=("completed", "size"),
                 success_pct=("completed", lambda s: 100 * s.mean())))
FW_GAP = swh.groupby("firmware_version", observed=True).completed.mean() * 100
GEN_TAB = (swh.groupby("charger_generation", observed=True)
           .agg(attempts=("completed", "size"), success=("completed", "mean")))
_zero = swh.loc[swh.charged_2w_min == 0, "completed"]
_nonzero = swh.loc[swh.charged_2w_min > 0, "completed"]
ZERO_N, ZERO_SUCC, NONZERO_SUCC = int(len(_zero)), float(_zero.mean()), float(_nonzero.mean())
STAT_MONTH = (swh.groupby(["station_id", "month"], observed=True).completed.mean() * 100)

del swh
gc.collect()

_KEEP = ["rider_id", "station_id", "event_ts", "date", "month", "hour", "event_type", "completed",
         "net_price", "list_price_inr", "discount_inr", "amount_charged_inr",
         "energy_to_recharge_kwh", "queue_wait_sec", "tariff_code", "summer"]
sw = sw[[c for c in _KEEP if c in sw.columns]].copy()
gc.collect()
print(f"logic working set: swaps {sw.shape}, open figures {len(plt.get_fignums())}")

## 6. Q3 — What drives rider churn?

The obvious hypothesis — "riders leave because swaps fail" — does **not** survive a naive test.
We show the trap, then fix it with a rider-month panel, and only then quantify the effect.

In [ ]:
END = sw.event_ts.max()
rid = sw.groupby("rider_id", observed=True).agg(
    first_swap=("event_ts", "min"), last_swap=("event_ts", "max"),
    swaps=("completed", "sum"), attempts=("completed", "size")).reset_index()
rid["failures"] = rid.attempts - rid.swaps
rid = rid.merge(riders[["rider_id", "signup_date", "plan_type", "signup_channel",
                        "vehicle_class", "home_city", "partner_id"]], on="rider_id", how="left")
rid["days_since_last"] = (END - rid.last_swap).dt.days
rid["churned_60d"] = (rid.days_since_last > 60).astype(int)
rid["fail_rate"] = rid.failures / rid.attempts

print(f"riders with at least one attempt: {len(rid):,} of {len(riders):,} registered")
print(f"median swaps per rider: {rid.swaps.median():.0f}   "
      f"median failures: {rid.failures.median():.0f}")
print(f"churned (no swap in last 60 days): {100*rid.churned_60d.mean():.1f}%")

### 6.1 Retention curves by signup cohort
Share of a signup cohort still swapping in month *k* after signup. Retention is remarkably
stable across cohorts — roughly 50% at month 12 — which means churn is not a cohort-quality
problem that is quietly improving or worsening.

In [ ]:
rid["cohort"] = rid.signup_date.dt.to_period("M")
act = sw[sw.completed == 1][["rider_id", "event_ts"]].merge(
    rid[["rider_id", "cohort"]], on="rider_id", how="inner")
act["k"] = ((act.event_ts.dt.year - act.cohort.dt.year) * 12 +
            (act.event_ts.dt.month - act.cohort.dt.month)).astype("int16")
cohort_n = rid.groupby("cohort").size()
ret = (act.groupby(["cohort", "k"]).rider_id.nunique().unstack().div(cohort_n, axis=0) * 100)
ret_show = ret.dropna(thresh=3).iloc[::2, :13].round(1)
table(ret_show, "Retention by signup cohort, % of cohort swapping in month k")

fig, axes_q = plt.subplots(1, 2, figsize=(12.8, 4.4), gridspec_kw={"width_ratios": [1.15, 1]})
ax = axes_q[0]
sel = [c for c in ret.index if str(c) >= "2023-09"][::3]
cmap = plt.cm.viridis(np.linspace(0.15, 0.85, len(sel)))
for c, col in zip(sel, cmap):
    r = ret.loc[c].dropna()
    ax.plot(r.index, r.values, color=col, lw=1.9, marker="o", ms=3, label=str(c))
ax.set_xlabel("months since signup"); ax.set_ylabel("% of cohort still swapping")
ax.set_title("Retention curves are stable — no cohort is collapsing")
ax.legend(ncol=2, fontsize=7.5, loc="upper right")
ax.set_ylim(30, 105)

ax = axes_q[1]
ch = rid.groupby("plan_type").agg(churn=("churned_60d", lambda s: 100 * s.mean()),
                                  n=("rider_id", "size"))
ch2 = rid.groupby("home_city").agg(churn=("churned_60d", lambda s: 100 * s.mean()),
                                   n=("rider_id", "size")).sort_values("churn")
yy = np.arange(len(ch2))
ax.barh(yy, ch2.churn, color=C_MAIN, height=0.6)
ax.axvline(rid.churned_60d.mean() * 100, color=C_BAD, ls="--", lw=1.2)
ax.set_yticks(yy); ax.set_yticklabels(ch2.index)
ax.set_xlabel("churn rate (% no swap in final 60 days)")
ax.set_title("Churn is flat across cities and plans (dashed = network avg)")
for i, (v, n) in enumerate(zip(ch2.churn, ch2.n)):
    ax.text(v + 0.4, i, f"{v:.0f}%", va="center", fontsize=8, color=MUTED)
fig.tight_layout()
savefig(fig, "q3_retention", "Cohorts = signup month; churn = no completed swap in the final 60 days.")
plt.show()

within_plan = rid.groupby("plan_type").churned_60d.mean() * 100
record("churn", {
    "churn_pct_60d": round(100 * float(rid.churned_60d.mean()), 1),
    "churn_by_plan": {k: round(float(v), 1) for k, v in within_plan.items()},
    "churn_spread_city_pp": round(float(ch2.churn.max() - ch2.churn.min()), 1),
    "retention_month12": round(float(ret[12].median(skipna=True)), 1),
})

### 6.2 The trap: the naive failure→churn link runs backwards
Riders with **zero** failed attempts churn *more* (51%) than riders with a 10–20% failure rate
(30%). The reason is that "zero failures" is dominated by riders who made a handful of attempts
and left — their failure rate is undefined-but-zero. Activity level, not reliability, is doing
the work. Any model that skips this will recommend the wrong fix.

In [ ]:
rid["fail_bucket"] = pd.cut(rid.fail_rate, [-0.001, 0, 0.05, 0.10, 0.20, 0.40, 1.0],
                            labels=["0%", "0–5%", "5–10%", "10–20%", "20–40%", "40%+"])
rid["vol_band"] = pd.qcut(rid.attempts, [0, .25, .5, .75, 1.0],
                          labels=["Q1 fewest", "Q2", "Q3", "Q4 most"])
naive = rid.groupby("fail_bucket", observed=True).agg(
    riders=("rider_id", "size"), churn_pct=("churned_60d", lambda s: 100 * s.mean()),
    median_attempts=("attempts", "median"))
table(naive.round(1), "NAIVE view: churn by personal failed-attempt share")

pivot = (rid.pivot_table(index="fail_bucket", columns="vol_band", values="churned_60d",
                         aggfunc="mean", observed=True) * 100)
table(pivot.round(1), "Churn (%) within activity quartile — the relationship flips")

fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.3))
ax = axes[0]
ax.bar(np.arange(len(naive)), naive.churn_pct, color=MUTED, width=0.62)
ax.set_xticks(np.arange(len(naive)))
ax.set_xticklabels(naive.index.astype(str), rotation=20)
ax.set_ylabel("churn (%)"); ax.set_ylim(0, 60)
ax.set_title("Naive: 'zero failures' riders churn most")
for i, v in enumerate(naive.churn_pct):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center", fontsize=8.5)

ax = axes[1]
q4 = pivot["Q4 most"].dropna()
q1 = pivot["Q1 fewest"].dropna()
ax.plot(np.arange(len(q4)), q4.values, color=C_BAD, lw=2.2, marker="o", label="most active quartile")
ax.plot(np.arange(len(q1)), q1.values, color=MUTED, lw=1.6, marker="o", ms=4,
        label="least active quartile")
ax.set_xticks(np.arange(len(pivot)))
ax.set_xticklabels(pivot.index.astype(str), rotation=20)
ax.set_ylabel("churn (%)")
ax.set_title("Stratified: among active riders, failures do predict churn")
ax.legend()
fig.tight_layout()
savefig(fig, "q3_simpson", "Attempts quartiles from swap_events; churn = no swap in final 60 days.")
plt.show()

### 6.3 The correct test: a rider-month panel
For every rider-month with activity, we measure that month's failure rate and whether the rider
returned in either of the next two months. This removes the activity-level artefact because every
observation is a rider who was demonstrably active that month.

In [ ]:
panel = (sw.groupby(["rider_id", "month"], observed=True)
         .agg(attempts=("completed", "size"), swaps=("completed", "sum"),
              abandoned=("event_type", lambda s: (s == "abandoned_queue").sum()),
              stockout=("event_type", lambda s: (s == "failed_no_charged_battery").sum()))
         .reset_index())
panel["fail_rate"] = 1 - panel.swaps / panel.attempts
months = sorted(sw.month.unique())
midx = {m: i for i, m in enumerate(months)}
panel["mi"] = panel.month.map(midx)
# set of (rider, month-index) pairs that actually exist -- NOT derived offsets, which would
# match a rider's own last month and make every rider look retained (a subtle but fatal bug)
active = set(zip(panel.rider_id.astype(str), panel.mi))
panel = panel[panel.mi <= max(midx.values()) - 2]
panel["returned"] = [(str(r), m + 1) in active or (str(r), m + 2) in active
                     for r, m in zip(panel.rider_id, panel.mi)]
panel["churn_next"] = 1 - panel.returned.astype(int)
print(f"rider-month observations: {len(panel):,}  (riders: {panel.rider_id.nunique():,})")
print(f"overall 2-month churn, conditional on being active: {100*panel.churn_next.mean():.1f}%")

panel["fb"] = pd.cut(panel.fail_rate, [-0.001, 0.0001, 0.05, 0.10, 0.20, 0.40, 1.0],
                     labels=["0%", "0–5%", "5–10%", "10–20%", "20–40%", "40%+"])
pm = panel.groupby("fb", observed=True).agg(
    rider_months=("churn_next", "size"), churn_pct=("churn_next", lambda s: 100 * s.mean()))
table(pm.round(2), "2-month churn by that month's swap failure rate (rider-month panel)")

base = pm.loc["0%", "churn_pct"]
hi = 100 * panel.loc[panel.fail_rate >= 0.10, "churn_next"].mean()
lo = 100 * panel.loc[panel.fail_rate < 0.10, "churn_next"].mean()
print(f"\nchurn when failure rate ≥10%:  {hi:.1f}%")
print(f"churn when failure rate <10%:  {lo:.1f}%   →  gap {hi-lo:+.1f} pp")

fig, ax = plt.subplots(figsize=(8.6, 4.2))
ax.bar(np.arange(len(pm)), pm.churn_pct, color=[C_GOOD, C_GOOD, C_GOOD, C_WARN, C_BAD, C_BAD],
       width=0.62)
for i, (v, n) in enumerate(zip(pm.churn_pct, pm.rider_months)):
    ax.text(i, v + 0.6, f"{v:.1f}%", ha="center", fontsize=9, color=INK)
    ax.text(i, 1.0, f"n={n:,.0f}", ha="center", fontsize=7.4, color="white")
ax.set_xticks(np.arange(len(pm))); ax.set_xticklabels(pm.index.astype(str))
ax.set_xlabel("swap failure rate in the month"); ax.set_ylabel("churn over next 2 months (%)")
ax.set_title("Conditional on being active, a bad month raises churn ~11 pp")
fig.tight_layout()
savefig(fig, "q3_panel_churn", "Rider-month panel; churn = no swap in either of the next two months.")
plt.show()

record("churn_panel", {
    "rider_months": int(len(panel)),
    "churn_zero_fail_pct": round(float(base), 1),
    "churn_ge10_fail_pct": round(float(hi), 1),
    "churn_lt10_fail_pct": round(float(lo), 1),
    "gap_pp": round(float(hi - lo), 1),
    "naive_zero_fail_churn_pct": round(float(naive.loc["0%", "churn_pct"]), 1),
})

### 6.4 What a failed swap costs in riders
Converted to retention terms: the extra churn on a bad month, applied to the rider-months that
experienced ≥10% failure rates, gives the number of rider-months at risk — the population the
network's reliability problem is actually pushing out.

In [ ]:
bad_rm = int((panel.fail_rate >= 0.10).sum())
avg_swaps_per_rider_month = panel.swaps.mean()
lost_rider_months = bad_rm * (hi - lo) / 100
print(f"rider-months with ≥10% failure: {bad_rm:,}")
print(f"mean swaps per rider-month: {avg_swaps_per_rider_month:.1f}")
print(f"excess churn attributable to high-failure months: ≈{lost_rider_months:,.0f} rider-months")
avg_swaps_per_month_network = sw.completed.sum() / len(months)
print(f"monthly network swaps: {avg_swaps_per_month_network:,.0f}  →  "
      f"≈{100*lost_rider_months*avg_swaps_per_rider_month/avg_swaps_per_month_network:.1f}% of monthly volume at risk")

rec = sw[sw.completed == 1].groupby("net_price").size()
avg_margin = (sw[sw.completed == 1].amount_charged_inr
              - sw[sw.completed == 1].energy_to_recharge_kwh
              * sw[sw.completed == 1].station_id.map(stations.set_index("station_id").grid_tariff_inr_kwh)
              ).mean()
record("churn_value", {
    "lost_rider_months": round(float(lost_rider_months), 0),
    "swap_volume_at_risk_pct": round(float(100 * lost_rider_months * avg_swaps_per_rider_month
                                          / avg_swaps_per_month_network), 2),
    "avg_gm_per_swap": round(float(avg_margin), 2),
})

## 7. Q4 — What erodes margin?

VoltRelay's P&L per swap has four levers: list price, discount, energy cost, and fixed site cost
per swap. Only one of them is behaving badly.

In [ ]:
ok_sw = sw[sw.completed == 1].copy()
ok_sw = ok_sw.merge(stations[["station_id", "city", "grid_tariff_inr_kwh", "monthly_rent_inr",
                              "monthly_maintenance_inr", "expansion_wave", "slots_2w",
                              "slots_3w"]], on="station_id", how="left")
ok_sw["energy_cost"] = ok_sw.energy_to_recharge_kwh * ok_sw.grid_tariff_inr_kwh
ok_sw["gm"] = ok_sw.amount_charged_inr - ok_sw.energy_cost

window_months = len(months)
station_fixed_month = (stations.set_index("station_id")[["monthly_rent_inr",
                                                         "monthly_maintenance_inr"]]
                       .sum(axis=1).rename("fixed_month"))
st_sw = ok_sw.groupby("station_id", observed=True).agg(
    swaps=("completed", "size"), revenue=("amount_charged_inr", "sum"),
    list_value=("list_price_inr", "sum"), discount=("discount_inr", "sum"),
    energy=("energy_cost", "sum"), gm=("gm", "sum")).join(station_fixed_month)
st_sw["fixed_window"] = st_sw["fixed_month"] * window_months
st_sw = st_sw.join(stations.set_index("station_id")[["city", "zone", "expansion_wave",
                                                     "slots_2w", "commissioned_date"]])
st_sw["gm_per_swap"] = st_sw.gm / st_sw.swaps
st_sw["fixed_per_swap"] = st_sw.fixed_window / st_sw.swaps
st_sw["contribution_per_swap"] = st_sw.gm_per_swap - st_sw.fixed_per_swap
st_sw["contribution"] = st_sw.gm - st_sw.fixed_window
st_sw["swaps_per_month"] = st_sw.swaps / window_months

unit = pd.Series({
    "list price per swap": ok_sw.list_price_inr.mean(),
    "less discounts": -ok_sw.discount_inr.mean(),
    "= net revenue per swap": ok_sw.amount_charged_inr.mean(),
    "less energy cost": -ok_sw.energy_cost.mean(),
    "= gross margin per swap": ok_sw.gm.mean(),
    "less fixed site cost per swap": -st_sw.fixed_window.sum() / st_sw.swaps.sum(),
    "= contribution per swap": (ok_sw.gm.sum() - st_sw.fixed_window.sum()) / st_sw.swaps.sum(),
})
table(unit.round(2).to_frame("INR per swap"), "Unit economics per completed swap")

record("unit_economics", {k: round(float(v), 2) for k, v in unit.items()})
record("margin_totals", {
    "revenue": float(ok_sw.amount_charged_inr.sum()),
    "gross_margin": float(ok_sw.gm.sum()),
    "fixed_cost": float(st_sw.fixed_window.sum()),
    "contribution": float(ok_sw.gm.sum() - st_sw.fixed_window.sum()),
    "list_value": float(ok_sw.list_price_inr.sum()),
    "discounts": float(ok_sw.discount_inr.sum()),
    "discount_pct_of_list": round(100 * float(ok_sw.discount_inr.sum() / ok_sw.list_price_inr.sum()), 2),
})

### 7.1 The real margin problem is fixed cost spread over too few swaps
23 of 152 stations lose money on a contribution basis. Every one of them is a low-throughput site
carrying ₹15k–₹28k/month of rent and maintenance. This is an *asset utilisation* problem, not a
pricing problem.

In [ ]:
neg = st_sw[st_sw.contribution_per_swap < 0].sort_values("contribution")
print(f"stations with negative contribution: {len(neg)} of {len(st_sw)}")
print(f"total monthly contribution loss at those stations: "
      f"{inr(neg.contribution.sum()/window_months)}")
table(neg[["city", "zone", "expansion_wave", "swaps_per_month", "gm_per_swap",
           "fixed_per_swap", "contribution_per_swap"]].head(12).round(1),
      "Worst 12 stations by contribution per swap")

by_wave = st_sw.groupby("expansion_wave").agg(
    stations=("swaps", "size"), swaps_per_month=("swaps_per_month", "median"),
    gm_per_swap=("gm_per_swap", "median"), fixed_per_swap=("fixed_per_swap", "median"),
    contribution_per_swap=("contribution_per_swap", "median"))
table(by_wave.round(2), "By expansion wave — new sites are not yet through the utilisation ramp")

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.5))
ax = axes[0]
for wave, c in [("Launch", C_MAIN), ("Wave1_2024H2", C_WARN), ("Wave2_2025H1", C_BAD)]:
    d = st_sw[st_sw.expansion_wave == wave]
    ax.scatter(d.swaps_per_month, d.contribution_per_swap, s=34, alpha=0.8, color=c, label=wave)
ax.axhline(0, color=INK, lw=1)
ax.set_xscale("log")
ax.set_xlabel("swaps per month (log)"); ax.set_ylabel("contribution per swap (₹)")
ax.set_title("Below ~1,000 swaps/month, sites stop covering fixed cost")
ax.legend(title="expansion wave")

ax = axes[1]
srt = st_sw.sort_values("contribution")
cum = srt.contribution.cumsum()
ax.plot(np.arange(len(cum)), cum / 1e5, color=C_BAD, lw=2)
ax.axhline(0, color=INK, lw=1)
ax.set_xlabel("stations ranked worst → best"); ax.set_ylabel("cumulative contribution (₹ lakh)")
ax.set_title("Losses concentrate in ~25 sites, not spread evenly")
n_neg = int((st_sw.contribution < 0).sum())
ax.axvline(n_neg, color=MUTED, ls=":")
ax.text(n_neg + 3, cum.min() / 1e5 * 0.8, f"{n_neg} loss-making sites", fontsize=8.5, color=MUTED)
fig.tight_layout()
savefig(fig, "q4_station_margin", "Fixed cost = (rent + maintenance) × 18 months; contribution excludes battery capex.")
plt.show()

record("margin_stations", {
    "negative_contribution_count": int(n_neg),
    "negative_contribution_monthly_inr": float(neg.contribution.sum() / window_months),
    "median_contribution_per_swap": round(float(st_sw.contribution_per_swap.median()), 2),
    "worst_station": str(neg.index[0]),
    "worst_contribution_per_swap": round(float(neg.contribution_per_swap.iloc[0]), 1),
})

### 7.2 Discount leakage: one contract amendment moves 2.4 pp of margin
Network discount depth jumps from 7.0% to 9.45% in November 2024 and never recovers. It is a
single partner's renegotiation, not a broad price war.

In [ ]:
disc_m = (sw.groupby("month", observed=True)
          .agg(list_value=("list_price_inr", "sum"), discount=("discount_inr", "sum"),
               swaps=("completed", "sum")))
disc_m["discount_pct"] = 100 * disc_m.discount / disc_m.list_value
disc_m.index = disc_m.index.astype(str)

pv = partners[partners.amendment_date.notna()].iloc[0]
print(f"contract amendment: {pv.partner_name} ({pv.partner_id}) on {pv.amendment_date.date()}, "
      f"discount {pv.discount_pct}% → {pv.discount_pct_after_amendment}%")

disc_partner = ok_sw.merge(riders[["rider_id", "partner_id"]], on="rider_id", how="left")
disc_partner["partner_id"] = disc_partner.partner_id.fillna("indie")
pt = disc_partner.groupby("partner_id").agg(
    swaps=("completed", "size"), list_value=("list_price_inr", "sum"),
    discount=("discount_inr", "sum"), gm=("gm", "sum"))
pt["discount_pct"] = 100 * pt.discount / pt.list_value
pt["gm_per_swap"] = pt.gm / pt.swaps
pt = pt.join(partners.set_index("partner_id")[["partner_name", "contract_type", "discount_pct"]]
             .rename(columns={"discount_pct": "contract_discount"}))
table(pt.sort_values("discount_pct", ascending=False).round(2).head(8),
      "Discount depth by partner — the amended contract is the outlier")

leak = disc_m.discount.sum() - disc_m.list_value.sum() * 0.0702
print(f"\nexcess discount vs the pre-Nov-2024 7.0% norm: {inr(leak)} over 8 months "
      f"({inr(leak/8)}/month)")

fig, ax = plt.subplots(figsize=(9.6, 4.0))
ax.plot(np.arange(len(disc_m)), disc_m.discount_pct, color=C_BAD, lw=2.2, marker="o", ms=4)
ax.axhline(7.0, color=MUTED, ls=":")
ax.axvline(list(disc_m.index).index("2024-11"), color=C_WARN, ls="--", lw=1.4)
ax.annotate("partner contract amendment\n(10% → 28% on one partner)", (10.2, 8.2),
            fontsize=8.5, color=C_WARN)
ax.set_xticks(np.arange(len(disc_m))[::3])
ax.set_xticklabels([m[2:] for m in disc_m.index][::3])
ax.set_ylabel("discount as % of list price")
ax.set_title("Discount depth steps up in Nov-2024 and stays there")
fig.tight_layout()
savefig(fig, "q4_discount", "discount_inr / list_price_inr across completed and failed attempts.")
plt.show()

AVG_GM = float(ok_sw.gm.mean())
AVG_NET = float(ok_sw.amount_charged_inr.mean())
del ok_sw, disc_partner
gc.collect()

record("discount", {
    "pre_nov24_pct": round(float(disc_m.discount_pct.iloc[:10].mean()), 2),
    "post_nov24_pct": round(float(disc_m.discount_pct.iloc[10:].mean()), 2),
    "excess_discount_inr": float(leak),
    "amended_partner": str(pv.partner_name),
    "amended_discount": f"{pv.discount_pct}% → {pv.discount_pct_after_amendment}%",
})

## 8. Q5 — Anomalies worth flagging separately

Not everything unusual is a problem. We separate **structural anomalies** (a cohort or site that
behaves differently for a reason), **data anomalies** (artefacts that must not be read as
performance), and **event windows** (short periods that distort period averages).

### 8.1 Station outliers, tested rather than eyeballed
A station with 30,000 attempts and a 91% success rate is not noise. We test each station against
the network rate using a binomial z-score, then correct for multiple comparisons.

In [ ]:
p_bar = sw.completed.mean()
stat_tab["z"] = ((stat_tab.success_pct / 100 - p_bar) /
                 np.sqrt(p_bar * (1 - p_bar) / stat_tab.attempts))
from math import erfc
stat_tab["p_value"] = [erfc(abs(z) / np.sqrt(2)) for z in stat_tab.z]
out = stat_tab.sort_values("z").head(8)[["station_id", "city", "zone", "attempts", "success_pct",
                                         "z", "p_value", "no_batt_pct", "abandon_pct",
                                         "charger_generation", "connectivity_tier"]]
table(out.reset_index(drop=True).round(4), "Stations significantly below network success rate")

steady = stat_tab.sort_values("z").head(8)
late = (STAT_MONTH[STAT_MONTH.index.get_level_values("station_id")
                   .isin(set(steady.station_id))].unstack())
late.columns = [str(c)[2:] for c in late.columns]
table(late.round(1), "Are the worst stations persistently bad, or was it one bad window?")
_out = out.head(8)
print(f"All {len(_out)} of the worst stations are {_out.charger_generation.mode().iloc[0]} cabinets, "
      f"and all {len(_out)} sit in {', '.join(sorted(set(_out.city)))} — the oldest launch-wave cities. "
      "Their annual average hides the story: they hold ~95.5% in normal months and fall to 79–88% "
      "every April–June.")

record("station_outliers", {
    "worst_station": str(out.station_id.iloc[0]),
    "worst_success_pct": round(float(out.success_pct.iloc[0]), 2),
    "z": round(float(out.z.iloc[0]), 1),
    "all_gen1": bool((out.charger_generation == "Gen1").all()),
})

### 8.2 The Kyron battery cohort — a single-supplier asset problem
Kyron packs sit at a median 61% SOH versus 80% for the other two suppliers, on *fewer* deliveries.
1,385 of 1,438 end-of-life retirements are Kyron, and every retirement is dated 15 June 2025 —
a bulk write-off, not a gradual attrition. That is an anomaly with a P&L consequence.

In [ ]:
ky = age_deg.copy()
ky["retired"] = ky.retired_date.notna()
ret_date = batteries.retired_date.dropna().unique()
print(f"distinct retirement dates in the fleet: {list(ret_date)}")
print(f"packs retired: {int(batteries.retired_date.notna().sum()):,}")
table(batteries[batteries.retired_date.notna()].groupby(["supplier", "retirement_reason"]).size()
      .to_frame("packs"), "Retirements by supplier")

soh_by_lot = (age_deg.groupby(["supplier", "manufacturing_lot"])
              .agg(packs=("drop", "size"), median_soh=("current_soh_pct", "median"),
                   soh_drop_per_100=("drop", "median"))
              .reset_index())
soh_by_lot["soh_drop_per_100"] = (soh_by_lot.soh_drop_per_100 /
                                  age_deg.groupby(["supplier", "manufacturing_lot"]).deliv.median().values * 100)
lot_view = soh_by_lot[soh_by_lot.packs >= 40].sort_values("median_soh")
table(pd.concat([lot_view.head(6), lot_view.tail(4)]).round(2).reset_index(drop=True),
      "Lots with ≥40 packs, ranked by current SOH")

below70 = batteries[batteries.current_soh_pct < 70]
print(f"\npacks already below 70% SOH: {len(below70):,} "
      f"({100*len(below70)/len(batteries):.1f}% of fleet), "
      f"{int((below70.supplier == 'Kyron').sum())} of them Kyron")
record("kyron", {
    "packs": int((batteries.supplier == "Kyron").sum()),
    "median_soh": round(float(batteries[batteries.supplier == "Kyron"].current_soh_pct.median()), 1),
    "other_median_soh": round(float(batteries[batteries.supplier != "Kyron"].current_soh_pct.median()), 1),
    "retired_packs": int(batteries.retired_date.notna().sum()),
    "retired_kyron": int(((batteries.supplier == "Kyron") & batteries.retired_date.notna()).sum()),
    "retirement_date": "2025-06-15",
    "below_70_soh_packs": int(len(below70)),
})

### 8.3 Firmware: a real but *secondary* difference
Success differs by station firmware (v3.2.0 vs v3.2.1). But firmware was never rolled out during
the window — every station has one firmware for all 18 months, and `stations.csv` claims update
dates of Mar/Apr 2025 that never appear in the event log. So this is a cross-sectional hardware
difference, not a before/after effect. We hold generation and city constant before reading it.

In [ ]:
table(FW_TABLE.round(2), "Success by firmware within charger generation")
fw_gap_raw = FW_GAP
print(f"\nraw gap v3.2.1 − v3.2.0: {fw_gap_raw['v3.2.1'] - fw_gap_raw['v3.2.0']:+.2f} pp")
print("stations.csv firmware_updated_date (2025-03-10 / 2025-04-14) has no counterpart in the "
      "event log: station_firmware never changes for any of the 152 stations. Treat firmware as a "
      "fixed station attribute and do not read a rollout effect into it.")
record("firmware", {
    "raw_gap_pp": round(float(fw_gap_raw["v3.2.1"] - fw_gap_raw["v3.2.0"]), 2),
    "note": "cross-sectional; no rollout observable in event log",
})

### 8.4 Event windows and data artefacts
Four things distort naive period comparisons and must be called out:

In [ ]:
ev = []
ev.append({"anomaly": "summer 2024 (Apr–Jun)", "size": int(sw[sw.summer & (sw.event_ts.dt.year == 2024)].shape[0]),
           "effect": f"success {100*sw[sw.summer & (sw.event_ts.dt.year==2024)].completed.mean():.1f}% vs 95.6% baseline",
           "reading": "structural: heat + charging capacity"})
ev.append({"anomaly": "summer 2025 (Apr–Jun)", "size": int(sw[sw.summer & (sw.event_ts.dt.year == 2025)].shape[0]),
           "effect": f"success {100*sw[sw.summer & (sw.event_ts.dt.year==2025)].completed.mean():.1f}%",
           "reading": "structural, aggravated by pack retirements"})
ev.append({"anomaly": "heat-alert days", "size": HEAT_ATTEMPTS,
           "effect": f"success {HEAT_SUCC:.1f}% vs {NOHEAT_SUCC:.1f}%",
           "reading": "primary driver"})
ev.append({"anomaly": "full-hour outages", "size": int((hourly.outage_minutes > 0).sum()),
           "effect": "60-minute outage hours", "reading": "rare; 308 station-hours of 1.49M"})
ev.append({"anomaly": "telemetry missing/partial", "size": int((hourly.telemetry_status != 'ok').sum()),
           "effect": f"{(hourly.telemetry_status != 'ok').mean()*100:.1f}% of station-hours",
           "reading": "blind spots concentrate in poor-connectivity sites"})
ev.append({"anomaly": "Kyron bulk retirement", "size": 1438, "effect": "all dated 2025-06-15",
           "reading": "fleet capacity removed in one day"})
ev.append({"anomaly": "offline_batch sync", "size": OFFLINE_ATTEMPTS,
           "effect": f"{OFFLINE_SHARE*100:.1f}% of attempts",
           "reading": "may under-record failures; check before trusting site comparisons"})
table(pd.DataFrame(ev), "Anomaly register")

table(OFF_TABLE.round(2), "Success by sync mode")
table(CONN_OFFLINE.round(2), "Offline-batch share by connectivity tier")

### 8.5 Time-window effects: what *doesn't* move the needle
Rain, public holidays, festivals and e-commerce sale days show no degradation — sale days actually
look *better*, because they cluster in the cool Sep–Nov months. Reporting these as "no effect after
controlling for season" is as useful as finding an effect.

In [ ]:
ctx_tab = CTX_TABLE
table(CTX_TABLE.sort_values("delta_pp"), "External conditions vs success rate")
record("external_conditions", CTX_TABLE.round(2).to_dict(orient="records"))

## 9. Deep dives

### 9.1 Pricing pilot: peak/off-peak tariffs from October 2024 (difference-in-differences)
56 stations start charging a PEAK tariff in peak hours and an OFFPEAK tariff outside them. Does
demand shift out of the expensive hours, and does total volume fall?

In [ ]:
peak_stations = sorted(sw[sw.tariff_code == "PEAK"].station_id.unique())
sw["peak_hours"] = sw.hour.isin(sw[sw.tariff_code == "PEAK"].hour.unique())
first_peak = sw[sw.tariff_code == "PEAK"].event_ts.min()
print(f"stations with a PEAK tariff: {len(peak_stations)} of {sw.station_id.nunique()}")
print(f"first PEAK event: {first_peak}")
print("peak hours in tariff design:", sorted(sw[sw.tariff_code == 'PEAK'].hour.unique()))

sw["treat_pricing"] = sw.station_id.isin(peak_stations)
wk = sw.event_ts.dt.to_period("W").astype(str)
sw["week"] = wk
panel_p = (sw.groupby(["station_id", "week", "treat_pricing"], observed=True)
           .agg(swaps=("completed", "sum"), attempts=("completed", "size"),
                peak_swaps=("completed", lambda s: np.nan)))
_pk = sw[sw.peak_hours]
panel_p["peak_swaps"] = (_pk.groupby(["station_id", "week"], observed=True).completed.sum()
                         .reindex(panel_p.index).fillna(0))
panel_p["peak_share"] = panel_p.peak_swaps / panel_p.swaps.replace(0, np.nan)
panel_p = panel_p.reset_index()
panel_p["post"] = panel_p.week >= "2024-10-01"
pre_weeks = panel_p[~panel_p.post].week.nunique()
print(f"weeks pre-pilot: {pre_weeks}, post: {panel_p[panel_p.post].week.nunique()}")

pre = panel_p[~panel_p.post].groupby("treat_pricing").swaps.mean()
post = panel_p[panel_p.post].groupby("treat_pricing").swaps.mean()
did_level = (post[True] - pre[True]) - (post[False] - pre[False])
log_did = float(np.log(post[True] / pre[True]) - np.log(post[False] / pre[False]))

# parallel-trends diagnostic: compare pre-period growth before the pilot
prep = panel_p[~panel_p.post]
h1 = prep[prep.week < "2024-05-15"]
h2 = prep[prep.week >= "2024-05-15"]
trends = {}
for tr in (True, False):
    a = h1[h1.treat_pricing == tr].groupby("week").swaps.mean().mean()
    b = h2[h2.treat_pricing == tr].groupby("week").swaps.mean().mean()
    trends[tr] = 100 * (b / a - 1)
print(f"\nPRE-PERIOD GROWTH (Jan-Apr vs May-Sep 2024), the parallel-trends test:")
print(f"  pilot stations : {trends[True]:+.1f}%")
print(f"  control stations: {trends[False]:+.1f}%")
print(f"  → pre-trend gap {trends[True]-trends[False]:+.1f} pp. The groups were NOT on the same "
      f"trajectory before the pilot, so the level DiD is biased; the log DiD below is the more "
      f"defensible estimate.")
pre_s = panel_p[~panel_p.post].groupby("treat_pricing").peak_share.mean()
post_s = panel_p[panel_p.post].groupby("treat_pricing").peak_share.mean()
did_share = (post_s[True] - pre_s[True]) - (post_s[False] - pre_s[False])
print(f"\nlevel DiD on weekly swaps per station: {did_level:+.2f} "
      f"({100*did_level/pre[True]:+.2f}% of treated baseline) — biased by the pre-trend gap")
print(f"log DiD (ratio of ratios)            : {100*(np.exp(log_did)-1):+.2f}%")
print(f"DiD on peak-hour share of swaps      : {100*did_share:+.2f} pp")

# bootstrap over stations for the level DiD
rng = np.random.default_rng(7)
boots = []
st_list = panel_p.station_id.unique()
treated_st = [s for s in st_list if s in set(peak_stations)]
control_st = [s for s in st_list if s not in set(peak_stations)]
for _ in range(400):
    ts = rng.choice(treated_st, len(treated_st), replace=True)
    cs = rng.choice(control_st, len(control_st), replace=True)
    sub = panel_p[panel_p.station_id.isin(set(set(ts) | set(cs)))]
    a = sub[(sub.treat_pricing) & (~sub.post)].swaps.mean()
    b = sub[(sub.treat_pricing) & (sub.post)].swaps.mean()
    c = sub[(~sub.treat_pricing) & (~sub.post)].swaps.mean()
    d = sub[(~sub.treat_pricing) & (sub.post)].swaps.mean()
    boots.append(np.log(b / a) - np.log(d / c))
lo, hi = np.nanpercentile(boots, [2.5, 97.5])
print(f"bootstrap 95% CI on the log DiD: [{100*(np.exp(lo)-1):+.1f}%, {100*(np.exp(hi)-1):+.1f}%]")

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.2))
ax = axes[0]
for tr, c, lab in [(True, C_BAD, f"pilot ({len(treated_st)} stations)"), (False, MUTED, f"control ({len(control_st)})")]:
    d = (panel_p[panel_p.treat_pricing == tr].groupby("week").swaps.mean())
    idx = np.arange(len(d))
    base = d.iloc[:8].mean()
    ax.plot(idx, 100 * d.values / base, color=c, lw=2 if tr else 1.5, label=lab)
ax.axvline(pre_weeks, color=C_WARN, ls="--", lw=1.3)
ax.text(pre_weeks + 2, 105, "peak/off-peak tariffs start", fontsize=8, color=C_WARN)
ax.set_xlabel("week index"); ax.set_ylabel("swaps per station (indexed to first 8 weeks)")
ax.set_title("Pilot stations grew slower after the peak surcharge — but pre-trends already differed")
ax.legend(loc="upper left")

ax = axes[1]
for tr, c, lab in [(True, C_BAD, "pilot"), (False, MUTED, "control")]:
    d = panel_p[panel_p.treat_pricing == tr].groupby("week").peak_share.mean() * 100
    ax.plot(np.arange(len(d)), d.values, color=c, lw=2 if tr else 1.5, label=lab)
ax.axvline(pre_weeks, color=C_WARN, ls="--", lw=1.3)
ax.set_xlabel("week index"); ax.set_ylabel("% of swaps in peak hours")
ax.set_title("Peak-hour share barely moves: riders do not shift out of peak")
ax.legend(loc="upper right")
fig.tight_layout()
savefig(fig, "q9_pricing_did", "DiD, pre 2024-01→2024-09, post 2024-10→2025-06, 152 stations.")
plt.show()

record("pricing_pilot", {
    "treated_stations": len(treated_st),
    "did_level_swaps": round(float(did_level), 2),
    "did_level_pct": round(float(100 * did_level / pre[True]), 2),
    "log_did_pct": round(float(100 * (np.exp(log_did) - 1)), 2),
    "log_did_ci_pct": [round(float(100 * (np.exp(lo) - 1)), 1),
                       round(float(100 * (np.exp(hi) - 1)), 1)],
    "did_peak_share_pp": round(float(100 * did_share), 2),
    "pre_trend_pilot_pct": round(float(trends[True]), 1),
    "pre_trend_control_pct": round(float(trends[False]), 1),
    "peak_hours": sorted(int(h) for h in sw[sw.tariff_code == "PEAK"].hour.unique()),
})

### 9.2 Competitor entry: 10 stations since 15 Jan 2025 (difference-in-differences)
If competition bites, exposed stations lose volume relative to neighbours. They don't.

In [ ]:
comp = stations.loc[stations.competitor_within_1_5km_since.notna(), "station_id"].tolist()
sw["exposed"] = sw.station_id.isin(comp)
p2 = (sw.groupby(["station_id", "week", "exposed"], observed=True)
      .agg(swaps=("completed", "sum"), riders=("rider_id", "nunique")).reset_index())
p2["post"] = p2.week >= "2025-01-15"
p2 = p2[p2.week >= "2024-07-15"]
q_pre = p2[~p2.post].groupby("exposed").swaps.mean()
q_post = p2[p2.post].groupby("exposed").swaps.mean()
did2 = (q_post[True] - q_pre[True]) - (q_post[False] - q_pre[False])
log_did2 = float(np.log(q_post[True] / q_pre[True]) - np.log(q_post[False] / q_pre[False]))
did2_pct = 100 * did2 / q_pre[True]
print(f"exposed stations: {len(comp)}")
print(f"level DiD: {did2:+.2f} swaps/station/week ({did2_pct:+.2f}%)")
print(f"log DiD  : {100*(np.exp(log_did2)-1):+.2f}%")

_b = []
_ex = [x for x in p2.station_id.unique() if x in set(comp)]
_ct = [x for x in p2.station_id.unique() if x not in set(comp)]
for _ in range(300):
    xs = rng.choice(_ex, len(_ex), replace=True)
    cs2 = rng.choice(_ct, len(_ct), replace=True)
    sub = p2[p2.station_id.isin(set(set(xs) | set(cs2)))]
    a = sub[(sub.exposed) & (~sub.post)].swaps.mean()
    b = sub[(sub.exposed) & (sub.post)].swaps.mean()
    c = sub[(~sub.exposed) & (~sub.post)].swaps.mean()
    d = sub[(~sub.exposed) & (sub.post)].swaps.mean()
    _b.append(np.log(b / a) - np.log(d / c))
_lo2, _hi2 = np.nanpercentile(_b, [2.5, 97.5])
print(f"bootstrap 95% CI on the log DiD: [{100*(np.exp(_lo2)-1):+.1f}%, {100*(np.exp(_hi2)-1):+.1f}%]")

fig, ax = plt.subplots(figsize=(9.4, 4.0))
for ex, c, lab in [(True, C_BAD, "exposed (competitor <1.5 km)"), (False, MUTED, "control")]:
    d = p2[p2.exposed == ex].groupby("week").swaps.mean()
    ax.plot(np.arange(len(d)), d.values, color=c, lw=2 if ex else 1.5, label=lab)
ax.axvline(p2[~p2.post].week.nunique(), color=C_WARN, ls="--", lw=1.3)
ax.text(p2[~p2.post].week.nunique() + 2, ax.get_ylim()[1] * 0.96, "competitor opens", fontsize=8, color=C_WARN)
ax.set_xlabel("week index (from Jul-2024)"); ax.set_ylabel("swaps per station per week")
ax.set_title("Competitor entry: a small, statistically weak volume effect")
ax.legend(loc="lower right")
fig.tight_layout()
savefig(fig, "q9_competitor_did", "10 exposed vs 142 control stations, DiD.")
plt.show()

record("competitor", {"exposed_stations": len(comp), "did_swaps": round(float(did2), 2),
                      "did_pct": round(float(did2_pct), 2),
                      "log_did_pct": round(float(100 * (np.exp(log_did2) - 1)), 2),
                      "log_did_ci_pct": [round(float(100 * (np.exp(_lo2) - 1)), 1),
                                        round(float(100 * (np.exp(_hi2) - 1)), 1)]})

### 9.3 Battery-bank solvency: the pack bank grew, then was cut in one day
Active packs are commissioned minus retired, and demand keeps growing. The ratio of packs to daily
swaps is the network's ability to put a charged battery in someone's hand. It is not a slow decline:
coverage was built steadily through 2024 and then fell sharply in June 2025 when 1,438 packs were
written off at once.

In [ ]:
mend = month_ends = pd.date_range("2024-01-31", "2025-06-30", freq="ME")
rows = []
for m in mend:
    active = batteries[(batteries.commission_date <= m) &
                       (batteries.retired_date.isna() | (batteries.retired_date > m))]
    ky = active[active.supplier == "Kyron"]
    perm = sw[sw.month == m.to_period("M")]
    d = perm.event_ts.dt.normalize().nunique()
    usable = active[active.current_soh_pct >= 70]
    rows.append({"month": m, "active_packs": len(active), "kyron_active": len(ky),
                 "usable_packs_ge70soh": len(usable),
                 "daily_swaps": perm.completed.sum() / max(d, 1),
                 "packs_per_100_daily_swaps": 100 * len(active) / max(perm.completed.sum() / max(d, 1), 1),
                 "usable_per_100_daily_swaps": 100 * len(usable) / max(perm.completed.sum() / max(d, 1), 1)})
solv = pd.DataFrame(rows)
solv["month"] = solv.month.dt.strftime("%Y-%m")
table(solv.round(1), "Battery bank coverage over time")

fig, ax = plt.subplots(figsize=(10.0, 4.3))
xw = np.arange(len(solv))
ax.plot(xw, solv.packs_per_100_daily_swaps, color=C_MAIN, lw=2.2, marker="o", ms=4,
        label="all active packs")
ax.plot(xw, solv.usable_per_100_daily_swaps, color=C_BAD, lw=2.2, marker="s", ms=4,
        label="packs ≥70% SOH (usable)")
ax.fill_between(xw, solv.usable_per_100_daily_swaps, solv.packs_per_100_daily_swaps,
                color=C_BAD, alpha=0.1)
ax.set_xticks(xw[::3]); ax.set_xticklabels(solv.month[::3])
ax.set_ylabel("packs per 100 daily swaps")
ax.set_title("Coverage peaked in Dec-2024 and fell back to Jan-2024 levels in June 2025")
ax.legend(loc="lower left")
fig.tight_layout()
savefig(fig, "q9_battery_solvency", "Active = commissioned and not retired; usable = current_soh_pct ≥ 70.")
plt.show()

_peak_i = int(solv.packs_per_100_daily_swaps.idxmax())
record("solvency", {
    "start_packs_per_100": round(float(solv.packs_per_100_daily_swaps.iloc[0]), 1),
    "peak_packs_per_100": round(float(solv.packs_per_100_daily_swaps.iloc[_peak_i]), 1),
    "peak_month": str(solv.month.iloc[_peak_i]),
    "end_packs_per_100": round(float(solv.packs_per_100_daily_swaps.iloc[-1]), 1),
    "active_may25": int(solv.active_packs.iloc[-2]),
    "active_jun25": int(solv.active_packs.iloc[-1]),
    "packs_removed_pct": round(100 * (1 - solv.active_packs.iloc[-1] / solv.active_packs.iloc[-2]), 1),
    "kyron_share_of_fleet": round(100 * float((batteries.supplier == "Kyron").mean()), 1),
})

## 11. Robustness checks and a self-review

Sections 4–10 make four claims that a critical reader should attack. Each is tested here rather
than defended in prose.

1. *"Heat is primary."* — the driver model has no station fixed effects, so it may be picking up
   station-level confounders, and its standard errors ignore within-station clustering.
2. *"Cooling is worth ₹X."* — the earlier sizing assumed the whole summer gap was heat, which is
   not identified: summer also carries demand growth.
3. *"Kyron degrades 3.25× faster per delivery."* — this ignores pack age. Older packs would look
   worse regardless of supplier.
4. *"Failures raise churn by 4.8pp."* — reported without an interval.

In [ ]:
import gc as _gc

# rebuild only what these checks need, then release it again
_slim = sw[["station_id", "hour", "month", "event_ts", "completed", "event_type", "rider_id"]].copy()
_slim["swap_hour"] = _slim.event_ts.dt.floor("h")
_slim = _slim.merge(hourly[["station_id", "hour_start", "cabinet_temp_c", "charged_2w_min"]]
                    .rename(columns={"hour_start": "swap_hour"}),
                    on=["station_id", "swap_hour"], how="left")
_slim["city"] = _slim.station_id.map(stations.set_index("station_id").city)
_slim["date"] = _slim.event_ts.dt.normalize()
_slim = _slim.merge(context[["city", "date", "heat_alert"]], on=["city", "date"], how="left")
_slim = _slim.drop(columns=["swap_hour"])
_slim["summer"] = _slim.month.dt.month.isin([4, 5, 6])
_slim["hot"] = (_slim.cabinet_temp_c > 40).astype("int8")
_slim["month_idx"] = ((_slim.event_ts.dt.year - 2024) * 12 + _slim.event_ts.dt.month).astype("int16")
print(f"robustness frame: {_slim.shape[0]:,} rows, {_slim.cabinet_temp_c.notna().mean()*100:.1f}% with telemetry")

### 11.1 Station fixed effects
Adding 152 station dummies identifies heat and temperature **within the same station** — every
station is compared only to itself. Station-level attributes (charger generation, connectivity)
are absorbed by those dummies and drop out, which is exactly right: they cannot be identified.

In [ ]:
_slim["tband"] = pd.cut(_slim.cabinet_temp_c, [-1, 25, 35, 40, 45, 50, 99]).astype(str)
_slim["sband"] = pd.cut(_slim.charged_2w_min, [-0.1, 0, 3, 6, 99]).astype(str)
_slim["hblock"] = pd.cut(_slim.hour, [-1, 5, 10, 15, 19, 23]).astype(str)

FE_KEYS = ["station_id", "tband", "sband", "hblock", "month_idx"]
_cells = (_slim.groupby(FE_KEYS, observed=True)
          .agg(attempts=("completed", "size"), failures=("completed", lambda s: int((1 - s).sum())))
          .reset_index())
_cells = _cells[_cells.attempts > 0]
print(f"cells: {len(_cells):,} covering {_cells.attempts.sum():,} attempts")


def _dummies(df, col, ref, keep_top=None):
    levels = sorted(df[col].astype(str).unique())
    if ref in levels:
        levels.remove(ref)
    if keep_top and len(levels) > keep_top:
        levels = levels[:keep_top]
    return [(df[col].astype(str) == v).astype(float).values.reshape(-1, 1) for v in levels], levels


_parts = [np.ones((len(_cells), 1))]
_names = ["intercept"]
for col, ref in [("tband", "(-1.0, 25.0]"), ("sband", "(-0.1, 0.0]"), ("hblock", "(-1.0, 5.0]")]:
    ds, lv = _dummies(_cells, col, ref)
    _parts += ds
    _names += [f"{col}={v}" for v in lv]
_st_ids = sorted(_cells.station_id.astype(str).unique())
_st_ref = _st_ids[0]
for v in _st_ids[1:]:
    _parts.append((_cells.station_id.astype(str) == v).astype(float).values.reshape(-1, 1))
    _names.append(f"station={v}")
_parts.append(_cells.month_idx.values.reshape(-1, 1).astype(float))
_names.append("month_idx")

X = np.hstack(_parts)
print(f"design matrix: {X.shape[0]:,} × {X.shape[1]}")
_b, _se = fit_logit(X, (_cells.failures / _cells.attempts).values, _cells.attempts.values)
fe_or = pd.DataFrame({"term": _names, "odds_ratio": np.exp(_b), "lo": np.exp(_b - 1.96 * _se),
                      "hi": np.exp(_b + 1.96 * _se)})
show_terms = ["tband=(25.0, 35.0]", "tband=(35.0, 40.0]", "tband=(40.0, 45.0]",
              "tband=(45.0, 50.0]", "tband=(50.0, 99.0]", "sband=(-0.1, 0.0]"]
_fe_view = fe_or[fe_or.term.isin(show_terms)].copy()
_fe_view["term"] = _fe_view.term.str.replace("tband=", "cabinet ").str.replace("sband=", "stock ")
table(_fe_view.round(3).reset_index(drop=True),
      "Within-station odds ratios (station fixed effects, 152 dummies)",
      "reference: cabinet ≤25 °C, stock = 0 is the empty-stock level vs 7+")
record("fe_model", {r.term: [round(r.odds_ratio, 3), round(r.lo, 3), round(r.hi, 3)]
                    for r in _fe_view.itertuples(index=False)})

### 11.2 How much of the summer gap is actually heat?
The earlier sizing credited **all** summer degradation to heat. Within the same summer months,
hot hours (cabinet > 40 °C) versus cool hours at the **same station** isolates the thermal effect
from seasonality and demand growth. This is the defensible number.

In [ ]:
_c = _slim.dropna(subset=["cabinet_temp_c"])
_pairs = (_c.groupby([_c.station_id, _c.hot], observed=True)
          .agg(attempts=("completed", "size"), succ=("completed", "mean")).reset_index())
_pv = _pairs.pivot(index="station_id", columns="hot", values=["attempts", "succ"])
_ok = _pv[("attempts", 1)].notna() & _pv[("attempts", 0)].notna() & \
      (_pv[("attempts", 1)] >= 300) & (_pv[("attempts", 0)] >= 300)
_gap = (_pv.loc[_ok, ("succ", 0)] - _pv.loc[_ok, ("succ", 1)])
_w = _pv.loc[_ok, ("attempts", 1)]
WITHIN_GAP_PP = float(100 * (_gap * _w).sum() / _w.sum())
HOT_ATTEMPTS = int((_c.hot == 1).sum())
print(f"stations with ≥300 attempts on both sides: {int(_ok.sum())} of 152")
print(f"weighted within-station success gap, cool vs hot hours: {WITHIN_GAP_PP:.2f} pp")
print(f"attempts in hot hours (cabinet > 40 °C): {HOT_ATTEMPTS:,} "
      f"({100*HOT_ATTEMPTS/len(_c):.1f}% of attempts with telemetry)")

_summer_gap = float(100 * (_slim[_slim.summer & (_slim.heat_alert == False)].completed.mean()
                           - _slim[_slim.summer & (_slim.heat_alert == True)].completed.mean()))
print(f"cross-sectional heat-alert gap inside summer months: {_summer_gap:.2f} pp")
print("\n→ the within-station thermal gap is materially smaller than the raw seasonal gap, which is "
      "why the earlier ₹ estimate was too generous.")

THERMAL_SWAPS_18M = HOT_ATTEMPTS * WITHIN_GAP_PP / 100
THERMAL_SWAPS_CAPTURE = THERMAL_SWAPS_18M * 0.7  # cooling will not recover the whole gap
print(f"upper-bound swaps recoverable: {THERMAL_SWAPS_18M:,.0f} over 18 months; "
      f"at a realistic 70% capture: {THERMAL_SWAPS_CAPTURE:,.0f}")
record("thermal_robust", {
    "within_station_gap_pp": round(WITHIN_GAP_PP, 2),
    "heat_alert_gap_within_summer_pp": round(_summer_gap, 2),
    "hot_hour_attempts": HOT_ATTEMPTS,
    "pct_attempts_hot": round(100 * HOT_ATTEMPTS / len(_c), 1),
    "upper_bound_swaps_18m": round(THERMAL_SWAPS_18M),
    "captured_70pct_swaps_18m": round(THERMAL_SWAPS_CAPTURE),
})

### 11.3 Cooling targets — *which* cabinets
Ranked by expected swaps saved at that specific station: attempts in hot hours × that station's
own cool-vs-hot success gap. This turns "cool the cabinets" into a work order.

In [ ]:
_targ = pd.DataFrame({
    "hot_attempts": _pv.loc[_ok, ("attempts", 1)],
    "cool_attempts": _pv.loc[_ok, ("attempts", 0)],
    "cool_success": 100 * _pv.loc[_ok, ("succ", 0)],
    "hot_success": 100 * _pv.loc[_ok, ("succ", 1)],
})
_targ["gap_pp"] = _targ.cool_success - _targ.hot_success
_targ["swaps_at_stake"] = _targ.hot_attempts * _targ.gap_pp / 100
_targ = _targ.join(stations.set_index("station_id")[["city", "zone", "charger_generation"]])
_targ = _targ.sort_values("swaps_at_stake", ascending=False)
table(_targ.head(12)[["city", "zone", "charger_generation", "cool_attempts", "hot_attempts",
                      "cool_success", "hot_success", "gap_pp", "swaps_at_stake"]].round(1),
      "Top 12 cooling targets by swaps at stake")
record("cooling_targets", _targ.head(12).reset_index().round(2).to_dict(orient="records"))

fig, ax = plt.subplots(figsize=(9.8, 4.6))
_t = _targ.head(12)[::-1]
_cols = [C_BAD if g == "Gen1" else C_WARN for g in _t.charger_generation]
ax.barh(np.arange(len(_t)), _t.swaps_at_stake, color=_cols, height=0.66)
ax.set_yticks(np.arange(len(_t)))
ax.set_yticklabels([f"{i.split('-')[0][4:]}·{i.split('-')[2]} ({z.split('-')[-1][:14]})"
                    for i, z in zip(_t.index, _t.zone)], fontsize=8)
ax.set_xlabel("swaps at stake over 18 months (hot-hour attempts × station thermal gap)")
ax.set_title("Cool these 12 cabinets first — red = Gen1")
fig.tight_layout()
savefig(fig, "q11_cooling_targets", "Hot = cabinet temp > 40 °C. Minimum 300 attempts on each side.")
plt.show()

del _slim, _cells, _parts, X, _pairs, _pv, _targ, _c
_gc.collect()

### 11.4 Kyron: does the degradation claim survive controlling for age?
If Kyron packs are simply older, "faster per delivery" is a mis-reading. Regressing current SOH on
deliveries **and** age separates the two.

In [ ]:
_bat = batteries.merge(_deliv.rename("deliveries"), left_on="battery_id", right_index=True, how="left")
_bat["deliveries"] = _bat.deliveries.fillna(0)
_bat["age_days"] = (pd.Timestamp("2025-06-30") - _bat.commission_date).dt.days
_bb = _bat.dropna(subset=["current_soh_pct", "age_days"])
_bb = _bb[_bb.deliveries > 0]
_sup_d = [( _bb.supplier == s).astype(float).values for s in ["Kyron", "Amptek"]]
Xb = np.column_stack([np.ones(len(_bb)), _bb.deliveries.values, _bb.age_days.values,
                      (_bb.pack_type == "3W_4.8kWh").astype(float).values, *_sup_d])
coef, *_ = np.linalg.lstsq(Xb, _bb.current_soh_pct.values, rcond=None)
resid = _bb.current_soh_pct.values - Xb @ coef
dof = len(_bb) - Xb.shape[1]
s2 = float(resid @ resid) / dof
cov = s2 * np.linalg.inv(Xb.T @ Xb)
seb = np.sqrt(np.diag(cov))
age_ctl = pd.DataFrame({
    "term": ["intercept", "deliveries (per 100)", "age_days (per 100)", "3W pack",
             "supplier: Kyron", "supplier: Amptek"],
    "coef": coef * np.array([1, 100, 100, 1, 1, 1]),
    "se": seb * np.array([1, 100, 100, 1, 1, 1]),
})
age_ctl["t"] = age_ctl.coef / age_ctl.se
table(age_ctl.round(3), "SOH (pp) regressed on deliveries and age — reference supplier is Cellora")

_ky = _bat[_bat.supplier == "Kyron"]
_oth = _bat[_bat.supplier != "Kyron"]
print(f"\nmedian commission date — Kyron {_ky.commission_date.median().date()}, "
      f"others {_oth.commission_date.median().date()}")
print(f"median age — Kyron {_ky.age_days.median()/30.4:.1f} months, "
      f"others {_oth.age_days.median()/30.4:.1f} months")
print(f"median deliveries — Kyron {_ky.deliveries.median():.0f}, others {_oth.deliveries.median():.0f}")
print("\nKyron coefficient is the age- and usage-adjusted SOH penalty. If it stays large and "
      "negative, the supplier claim holds.")
record("kyron_age_adjusted", {
    "kyron_coef_pp": round(float(coef[4]), 2),
    "kyron_t": round(float(age_ctl.t.iloc[4]), 2),
    "kyron_median_age_months": round(float(_ky.age_days.median() / 30.4), 1),
    "other_median_age_months": round(float(_oth.age_days.median() / 30.4), 1),
    "kyron_median_deliveries": int(_ky.deliveries.median()),
    "other_median_deliveries": int(_oth.deliveries.median()),
})

### 11.5 Churn: an interval, and sensitivity to the definition
The +4.8pp gap is bootstrapped over riders (clustered), and the whole analysis is re-run under
30-, 60- and 90-day churn definitions.

In [ ]:
_rng = np.random.default_rng(11)
_p = panel.copy()
_hi_mask = _p.fail_rate >= 0.10
_gaps = []
_rid = _p.rider_id.astype(str).values
_uniq = np.unique(_rid)
for _ in range(300):
    pick = _rng.choice(_uniq, len(_uniq), replace=True)
    sub = _p[_p.rider_id.astype(str).isin(set(pick))]
    g = (100 * sub.loc[sub.fail_rate >= 0.10, "churn_next"].mean()
         - 100 * sub.loc[sub.fail_rate < 0.10, "churn_next"].mean())
    _gaps.append(g)
_glo, _ghi = np.nanpercentile(_gaps, [2.5, 97.5])
print(f"churn gap (≥10% vs <10% failure): {FINDINGS['churn_panel']['gap_pp']:+.1f} pp, "
      f"rider-clustered bootstrap 95% CI [{_glo:+.1f}, {_ghi:+.1f}] pp")

_sens = []
for _d in (30, 60, 90):
    _ch = (END - rid.last_swap).dt.days > _d
    _sens.append({"churn definition": f"no swap in final {_d} days",
                  "churn rate %": round(100 * float(_ch.mean()), 1),
                  "high-failure riders churn %": round(100 * float(_ch[rid.fail_rate >= 0.10].mean()), 1),
                  "low-failure riders churn %": round(100 * float(_ch[rid.fail_rate < 0.10].mean()), 1)})
table(pd.DataFrame(_sens), "Churn definition sensitivity (whole-life view, all riders)")
record("churn_robust", {"gap_ci_pp": [round(float(_glo), 1), round(float(_ghi), 1)]})

### 11.6 Partner performance — the contract question in one table

In [ ]:
_psw = sw.merge(riders[["rider_id", "partner_id"]], on="rider_id", how="left")
_psw["partner_id"] = _psw.partner_id.fillna("independent")
_ptab = _psw.groupby("partner_id").agg(
    attempts=("completed", "size"), swaps=("completed", "sum"),
    riders=("rider_id", "nunique"))
_ptab["success_pct"] = 100 * _ptab.swaps / _ptab.attempts
_ptab = _ptab.join(partners.set_index("partner_id")[["partner_name", "contract_type",
                                                     "discount_pct", "payment_terms_days"]])
_ptab = _ptab.join(partners.set_index("partner_id")["discount_pct_after_amendment"])
_ptab["effective_discount"] = _ptab.discount_pct_after_amendment.fillna(_ptab.discount_pct)
table(_ptab.sort_values("effective_discount", ascending=False).round(2),
      "Partner performance and effective discount (independent riders = no contract)")
record("partners", _ptab.reset_index().round(2).to_dict(orient="records"))

print("\nBonferroni note for §8.1: with 152 station tests, the family-wise 5% threshold is "
      f"p < {0.05/152:.5f}. All eight flagged stations are far below it, so the station outliers are "
      "not a multiple-comparisons artefact.")

## 10. Q6 — What VoltRelay should prioritise

Each recommendation is sized in completed swaps and gross margin, using the observed effect from
this data. Sizing is deliberately conservative: it prices the *association* we measured, not a
causal guarantee.

In [ ]:
avg_gm = AVG_GM
avg_net = AVG_NET
base_rate = float(sw[~sw.summer].completed.mean())
rows = []

# 1 thermal -- sized from the WITHIN-STATION thermal gap (section 11.2), not the seasonal gap,
# which would over-credit heat with demand growth and seasonality
extra = THERMAL_SWAPS_CAPTURE
rows.append({"initiative": "Cool the 12 cabinets with the largest thermal gap (shade, active cooling, charge throttling)",
             "basis": f"within-station success gap {WITHIN_GAP_PP:.1f}pp between cool and >40°C hours; "
                      f"{HOT_ATTEMPTS:,} attempts in hot hours; 70% capture assumed",
             "swaps_recovered": extra, "margin_inr": extra * avg_gm})

# 2 stock: eliminate zero-stock swap hours
gain_zero = float((NONZERO_SUCC - ZERO_SUCC) * ZERO_N)
rows.append({"initiative": "Hold a minimum charged-pack floor (never let a cabinet hit zero stock)",
             "basis": f"{ZERO_N:,} attempts happen in zero-stock hours at {100*ZERO_SUCC:.1f}% success",
             "swaps_recovered": gain_zero, "margin_inr": gain_zero * avg_gm})

# 3 battery replacement
ky_ret = int(((batteries.supplier == "Kyron") & batteries.retired_date.notna()).sum())
rows.append({"initiative": f"Replace Kyron pack cohort ({ky_ret:,} packs written off) and re-bid the supply contract",
             "basis": f"Kyron median SOH {batteries[batteries.supplier=='Kyron'].current_soh_pct.median():.0f}% vs "
                      f"{batteries[batteries.supplier!='Kyron'].current_soh_pct.median():.0f}%; "
                      f"{len(below70):,} packs already below 70%",
             "swaps_recovered": np.nan, "margin_inr": np.nan})

# 4 Gen1 retrofit
gain_gen1 = float((GEN_TAB.loc["Gen2", "success"] - GEN_TAB.loc["Gen1", "success"])
                  * GEN_TAB.loc["Gen1", "attempts"])
rows.append({"initiative": "Retrofit Gen1 cabinets to Gen2 charging hardware",
             "basis": f"Gen1 {100*GEN_TAB.loc['Gen1','success']:.1f}% vs Gen2 "
                      f"{100*GEN_TAB.loc['Gen2','success']:.1f}% success",
             "swaps_recovered": gain_gen1, "margin_inr": gain_gen1 * avg_gm})

# 5 abandoned queue conversion
ab = sw[sw.event_type == "abandoned_queue"]
conv = 0.5 * len(ab)
rows.append({"initiative": "Virtual queue with wait-time guarantee and auto-hold of a charged pack",
             "basis": f"{len(ab):,} queue abandonments, median wait {ab.queue_wait_sec.median():.0f}s",
             "swaps_recovered": conv, "margin_inr": conv * avg_gm})

# 6 station triage
neg_monthly = float(neg.contribution.sum() / window_months)
rows.append({"initiative": f"Fix or exit the {len(neg)} loss-making stations (rent renegotiation, relocation, host change)",
             "basis": f"contribution {inr(neg.contribution_per_swap.median())}/swap vs "
                      f"{inr(st_sw.contribution_per_swap.median())} network median",
             "swaps_recovered": np.nan, "margin_inr": -neg_monthly})

# 7 discount
rows.append({"initiative": "Re-tier partner discounts; cap the amended contract's 28%",
             "basis": f"discount depth {disc_m.discount_pct.iloc[:10].mean():.1f}% → "
                      f"{disc_m.discount_pct.iloc[10:].mean():.1f}% after Nov-2024",
             "swaps_recovered": np.nan, "margin_inr": leak / 8})

prio = pd.DataFrame(rows)
# Anything with a swaps_recovered figure was sized over the whole 18-month window, so its margin
# must be divided by 18. Items with no swap figure are already expressed per month (discount
# leakage, station overspend). Deriving this from the data instead of string-matching the title
# avoids exactly the kind of silent error that mislabels an 18-month total as a monthly rate.
_window_items = prio.swaps_recovered.notna()
prio["margin_inr_per_month"] = np.where(_window_items,
                                        prio.margin_inr / window_months, prio.margin_inr)
prio["annualised_inr"] = prio.margin_inr_per_month * 12
prio = prio.sort_values("margin_inr_per_month", ascending=False)
table(prio.assign(swaps_recovered=prio.swaps_recovered.round(0),
                  margin_inr=prio.margin_inr.round(0),
                  margin_inr_per_month=prio.margin_inr_per_month.round(0),
                  annualised_inr=prio.annualised_inr.round(0)),
      "Priorities, sized in swaps and gross margin",
      "window-sized items divided by 18 months; annualised column is the comparable figure")
print(f"total annualised opportunity: {inr(prio.annualised_inr.sum())}")

_prio_out = prio.assign(swaps_recovered=prio.swaps_recovered.round(0),
                        margin_inr=prio.margin_inr.round(0),
                        margin_inr_per_month=prio.margin_inr_per_month.round(0),
                        annualised_inr=prio.annualised_inr.round(0))
_prio_out = _prio_out.where(pd.notna(_prio_out), None)
record("priorities", _prio_out.to_dict(orient="records"))

fig, ax = plt.subplots(figsize=(10.6, 4.6))
d = prio.dropna(subset=["margin_inr_per_month"]).sort_values("margin_inr_per_month")
yy = np.arange(len(d))
ax.barh(yy, d.margin_inr_per_month / 1e5, color=[C_GOOD if v > 0 else C_BAD for v in d.margin_inr_per_month],
        height=0.6)
ax.set_yticks(yy)
ax.set_yticklabels([t[:64] + ("…" if len(t) > 64 else "") for t in d.initiative], fontsize=8.5)
ax.set_xlabel("gross margin impact (₹ lakh per month)")
ax.set_title("Where the money is: reliability first, then the cost base")
for i, v in enumerate(d.margin_inr_per_month / 1e5):
    ax.text(v + (0.06 if v > 0 else -0.06), i, f"{v:+.1f}", va="center",
            ha="left" if v > 0 else "right", fontsize=8.5, color=INK)
fig.tight_layout()
savefig(fig, "q10_priorities", "Impact = observed effect × population exposed; association, not guaranteed causation.")
plt.show()

## Executive summary
The cell below prints the numbers that go into the report, and writes them to
`reports/key_findings.json` so the written report and this notebook cannot drift apart.

In [ ]:
print("=" * 78)
print("VOLTRELAY — HEADLINE NUMBERS")
print("=" * 78)
print(f"Window                 {HEAD['window']}")
print(f"Swap attempts          {HEAD['attempts']:,}   completed {HEAD['swaps']:,} "
      f"({HEAD['success_pct']}%)")
print(f"Revenue                {inr(HEAD['revenue'])}")
print(f"Active riders          {HEAD['riders_active']:,} of {HEAD['riders_total']:,} registered")
print(f"Station spread         {FINDINGS['station_spread']['best']}% best → "
      f"{FINDINGS['station_spread']['worst']}% worst")
print(f"Summer success         {FINDINGS['seasonality']['summer24_success_pct']}% (2024) / "
      f"{FINDINGS['seasonality']['summer25_success_pct']}% (2025) vs "
      f"{FINDINGS['seasonality']['nonsummer_success_pct']}% otherwise")
print(f"Heat days              {FINDINGS['heat']['heat_days_success_pct']}% vs "
      f"{FINDINGS['heat']['normal_days_success_pct']}%")
print(f"Failure odds           {FINDINGS['heat_odds_ratio_by_model']}")
print(f"Churn                  {FINDINGS['churn']['churn_pct_60d']}% baseline; "
      f"{FINDINGS['churn_panel']['gap_pp']} pp penalty for a bad month")
print(f"Unit economics         net {inr(FINDINGS['unit_economics']['= net revenue per swap'])} → "
      f"contribution {inr(FINDINGS['unit_economics']['= contribution per swap'])} per swap")
print(f"Loss-making stations   {FINDINGS['margin_stations']['negative_contribution_count']} of 152, "
      f"{inr(FINDINGS['margin_stations']['negative_contribution_monthly_inr'])}/month")
print(f"Discount leakage       {inr(FINDINGS['discount']['excess_discount_inr'])} over 8 months")
print("=" * 78)

out = ROOT / "reports" / "key_findings.json"
def _jsonable(o):
    if hasattr(o, "item"):
        return o.item()
    return str(o)


out.write_text(json.dumps(FINDINGS, indent=2, default=_jsonable))
print(f"\nwrote {out}")
print(f"figures in {FIG}: {len(list(FIG.glob('*.png')))}")